# Module 3: Building the Attention Mechanism

So far, we've seen how text is converted into tokens and how those tokens can be represented as vectors.

But there is still an important problem.

Each token currently has its **own representation**.

Consider:
> **“The bank said there was no network.”**

The word **“bank”** has a vector representation, and **“network”** has another vector representation.

But language is contextual.

When we encounter the word **“bank”**, we don't just want its isolated representation. We want a representation that can incorporate information from the other words around it.

This is where **attention** comes in.

In this module, we will build attention step by step.

We will begin with a very simple version:

1. Represent each token as a vector.
2. Compare token representations using a **dot product**.
3. Use those comparisons as **attention scores**.
4. Normalize the scores into **attention weights**.
5. Use the weights to create **context vectors**.
6. Introduce trainable **Query, Key, and Value** projections.
7. Add scaling and causal masking.
8. Build causal self-attention.
9. Extend it to multi-head attention.
    
We will not jump straight to the final implementation.

We will build the mechanism from the inside out.

## Install packages

In [1]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
!pip install -q numpy

Versions of packages at the time of making this notebook:

In [2]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("numpy version:", version("numpy"))

torch version: 2.15.0+cpu
numpy version: 2.5.4


## Part 1

### 3.1 Our Example

Consider the sentence:

"The bank said there was no network."

This is a simple sentence, but it contains several relationships between tokens.

For example:

- "bank" is related to "said"
- "said" introduces the information that follows
- "no" changes the meaning of "network"
- "no" and "network" have an important relationship

These relationships give us something meaningful to explore when we build attention.

At first, we won't use real learned embeddings.

Instead, we'll use small illustrative vectors so that we can focus on understanding how attention works.

In [3]:
tokens = [
    "The",
    "bank",
    "said",
    "there",
    "was",
    "no",
    "network"
]

print(tokens)

['The', 'bank', 'said', 'there', 'was', 'no', 'network']


### 3.2 Representing Tokens as Vectors

A neural network cannot directly perform mathematical operations on words such as `"bank"` or `"network"`.

We therefore represent each token using a vector.

For this exercise, each token will have a **3-dimensional representation**.

These numbers are deliberately simple and illustrative.

They are **not learned embeddings**, and the individual dimensions do not represent specific concepts.

Our goal is simply to have vectors that we can use to understand the mechanics of attention.

In [4]:
import torch

torch.manual_seed(123)

inputs = torch.tensor([
    [0.42, 0.18, 0.76],  # The
    [0.81, 0.34, 0.59],  # bank
    [0.63, 0.91, 0.27],  # said
    [0.37, 0.22, 0.68],  # there
    [0.74, 0.56, 0.31],  # was
    [0.29, 0.79, 0.47],  # no
    [0.58, 0.67, 0.39],  # network
])
print(inputs)

tensor([[0.4200, 0.1800, 0.7600],
        [0.8100, 0.3400, 0.5900],
        [0.6300, 0.9100, 0.2700],
        [0.3700, 0.2200, 0.6800],
        [0.7400, 0.5600, 0.3100],
        [0.2900, 0.7900, 0.4700],
        [0.5800, 0.6700, 0.3900]])


##### **What Does Our Tensor Represent?**

Our tensor contains 7 vectors, one for each token.

Because each vector has 3 dimensions, the shape is:

```text
7 x 3
```

So we can think of it as:

```text
             dimension
             1      2      3

The       [ 0.42,  0.18,  0.76 ]
bank      [ 0.81,  0.34,  0.59 ]
said      [ 0.63,  0.91,  0.27 ]
there     [ 0.37,  0.22,  0.68 ]
was       [ 0.74,  0.56,  0.31 ]
no        [ 0.29,  0.79,  0.47 ]
network   [ 0.58,  0.67,  0.39 ]
```

Before we calculate attention, let's confirm this with PyTorch.

##### **Inspect the Shape**

In [5]:
print("Input shape:", inputs.shape)

Input shape: torch.Size([7, 3])


#### 3.2.1: What Are We Trying to Calculate?

##### **Which Tokens Should “Bank” Pay Attention To?**

Let's focus on the token:
> **“bank”**

We want to ask:
> **How strongly should “bank” relate to each of the other tokens in the sentence?**

For example:
```text
bank -> The
bank -> bank
bank -> said
bank -> there
bank -> was
bank -> no
bank -> network
```

We need some numerical way to measure these relationships.

Our first idea will be simple:
> **Compare the vector for “bank” with the vector for each other token.**

But how do we compare two vectors?

This brings us to one of the most important operations in attention:

##### **The Dot Product**

##### **What Is a Dot Product?**

Suppose we have two vectors:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$ a = [a_1,a_2,a_3] $

and:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$ b = [b_1,b_2,b_3] $

Their dot product is calculated by:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$ a \cdot b = a_1b_1+a_2b_2+a_3b_3 $

In simple terms:
> **Multiply corresponding elements, then add the results together.**

For example:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$ [2,3,4] \cdot [5,1,2] $

becomes:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$(2 \times 5) + (3 \times 1) + (4 \times 2) $

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$ = 10 + 3 + 8 $

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$ = 21 $

So the dot product takes **two vectors** and produces **one number**.

That single number gives us a way to measure how strongly the two vectors align.

##### **Calculate a Dot Product Manually**

In [6]:
a = torch.tensor([2.0, 3.0, 4.0])
b = torch.tensor([5.0, 1.0, 2.0])

result = (
    a[0] * b[0]
    + a[1] * b[1]
    + a[2] * b[2]
)
print(result)

tensor(21.)


##### **What Did We Just Do?**

We started with:

```text
a = [2, 3, 4]

b = [5, 1, 2]
```

Then we multiplied corresponding elements:

```text
2 × 5 = 10
3 × 1 =  3
4 × 2 =  8
```

And added them:
```text
10 + 3 + 8 = 21
```

So:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$a \cdot b = 21 $

The important idea is that the dot product gives us **one number describing the interaction between two vectors**.

That is exactly what we need for our first version of an attention score.

##### **From Dot Product to Attention Score**

Now let's return to our sentence.

Our `"bank"` vector is:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$ [0.81,0.34,0.59] $

Our `"network"` vector is:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$ [0.58,0.67,0.39] $

We can compare them using dot product:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$ bank \cdot network $

which means:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;$ (0.81 \times 0.58) + (0.34 \times 0.67) + (0.59 \times 0.39) $

The resulting number will be our **attention score** for the pair.

We can calculate it manually first.

##### **Calculate the Score Manually**

In [7]:
bank = inputs[1]
network = inputs[6]

part_1 = bank[0] * network[0]
part_2 = bank[1] * network[1]
part_3 = bank[2] * network[2]

print("First dimension:", part_1)
print("\t\t+")
print("Second dimension:", part_2)
print("\t\t+")
print("Third dimension:", part_3)
print("-" * 35)

bank_network_score = part_1 + part_2 + part_3
print("Attention score:", bank_network_score)

First dimension: tensor(0.4698)
		+
Second dimension: tensor(0.2278)
		+
Third dimension: tensor(0.2301)
-----------------------------------
Attention score: tensor(0.9277)


##### **Let PyTorch Calculate the Same Dot Product**

We don't have to manually multiply and add every element ourselves.

PyTorch provides `torch.dot()`:
```python
torch.dot(a, b)
```

Let's use it with our `"bank"` and `"network"` vectors.

##### Using `torch.dot()`

In [8]:
bank_network_score = torch.dot(bank, network)
print(bank_network_score)

tensor(0.9277)


This should give us the same result as our manual calculation.

This is an important pattern throughout this module:
> **First understand the operation. Then use the framework's convenient implementation.**

##### What About the `@` Operator?

There is another way to write the same operation for two 1-dimensional vectors:

```python
bank @ network
```
The `@` operator performs matrix multiplication, and when both inputs are vectors, this produces their dot product.

So:
```python
torch.dot(bank, network)
```

and:
```python
bank @ network
```

give us the same result.

In [9]:
print("Bank network score (dot method):", torch.dot(bank, network))
print("Bank network score (@ method):  ", bank @ network)

Bank network score (dot method): tensor(0.9277)
Bank network score (@ method):   tensor(0.9277)


We now understand what is happening underneath the `@`.

#### 3.2.2: Calculate Attention Scores

##### **From One Comparison to Every Token**

We've calculated one attention score:
```text
bank -> network
```

But attention needs to compare `"bank"` with every token in the sentence.

So we want:

```text
bank -> The
bank -> bank
bank -> said
bank -> there
bank -> was
bank -> no
bank -> network
```

Each comparison produces one dot product and therefore one attention score.

Let's calculate them one at a time first.

##### **Calculate All Scores for “Bank”**

In [10]:
query = inputs[1] # "bank as the input token for the query"

attention_scores = torch.empty(inputs.shape[0]) # return a tensor filled with uninitialized data of size 7 (the number of rows from inputs)
print("Attention scores of the query (input token 'bank') filled with uninitialized data of size 7 (before dot product):\n", attention_scores, end="\n\n")
print("-" * 150, end="\n\n")

# Do a dot product of the query (input token 'bank') and each other token in the sequence
for i, x_i in enumerate(inputs):
    print(f"   query (input token 'bank')\t   @\t   x_{i+1} ({tokens[i]})")
    print(f"{query}   @   {x_i}\n")
    attention_scores[i] = torch.dot(query, x_i) # dot product (these are 1-dim vectors so no need to use transpose)
    print("-" * 150, end="\n\n")

print("Attention scores of the query (input token 'bank') after dot product:\n", attention_scores)    

Attention scores of the query (input token 'bank') filled with uninitialized data of size 7 (before dot product):
 tensor([1.3317e+30, 3.3620e-41, 0.0000e+00, 0.0000e+00, 0.0000e+00, 3.3627e-41,
        0.0000e+00])

------------------------------------------------------------------------------------------------------------------------------------------------------

   query (input token 'bank')	   @	   x_1 (The)
tensor([0.8100, 0.3400, 0.5900])   @   tensor([0.4200, 0.1800, 0.7600])

------------------------------------------------------------------------------------------------------------------------------------------------------

   query (input token 'bank')	   @	   x_2 (bank)
tensor([0.8100, 0.3400, 0.5900])   @   tensor([0.8100, 0.3400, 0.5900])

------------------------------------------------------------------------------------------------------------------------------------------------------

   query (input token 'bank')	   @	   x_3 (said)
tensor([0.8100, 0.3400, 0.5900])   

Now we have **one attention score for every token**.

##### **Inspect the Scores**

In [11]:
for token, score in zip(tokens, attention_scores):
    print(f"{token:8} {score:.4f}")

The      0.8498
bank     1.1198
said     0.9790
there    0.7757
was      0.9727
no       0.7808
network  0.9277


These are our first **raw attention scores**.

At this stage, they are not probabilities.

They simply tell us the relative strength of the dot-product interaction between `"bank"` and each token.

- Small dot product example: we get the dot product by doing element-wise multiplication of two vectors and summing them together:

In [12]:
element_wise_multiplied_values = torch.empty(inputs.shape[1])

for i, elem in enumerate(inputs[0]):
    element_wise_multiplied_values[i] = query[i] * elem    

print("Element-wise multiplied values:\t\t", element_wise_multiplied_values)
summed_values = element_wise_multiplied_values.sum()
print("Dot product result (manual dot method): ", summed_values)
print("Dot product result (PyTorch dot method):", torch.dot(query, inputs[0]))

Element-wise multiplied values:		 tensor([0.3402, 0.0612, 0.4484])
Dot product result (manual dot method):  tensor(0.8498)
Dot product result (PyTorch dot method): tensor(0.8498)


##### **The Bigger Picture**

We've now built the first part of attention:


token representations

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;
    
dot products

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

attention scores

But we've only calculated the scores for **one query token**: `"bank"`.

Eventually, every token will need to compare itself with every other token.

That will give us an entire **attention-score matrix**.

Before we calculate that matrix efficiently, however, we need to solve another problem:

> **Raw attention scores are difficult to interpret directly**.

For example, our scores might look something like:
```text
[2.1, 1.8, 1.5, 1.2, 1.9, 1.4, 2.0]
```

What we really want is a set of values that tells us how the model distributes its attention across the tokens.

That brings us to the next step:

**normalizing the attention scores with softmax.**

#### 3.2.3: From Attention Scores to Attention Weights

##### **Why Do We Need to Normalize the Scores?**

We currently have raw attention scores for the query "bank".

Conceptually, they look like:

```text
The       ->  score
bank      ->  score
said      ->  score
there     ->  score
was       ->  score
no        ->  score
network   ->  score
```

These scores tell us how strongly `"bank"` interacts with each token.

But there is a problem.

The scores are just arbitrary numbers.

For example:
```text
[2.1, 1.8, 1.5, 1.2, 1.9, 1.4, 2.0]
```

We would like to turn them into something easier to interpret:

- values between **0 and 1**
- all values adding up to **1**
- larger values representing stronger attention
- 
This gives us a kind of **attention distribution**.

We call these normalized values **attention weights**.

##### What Should the Weights Look Like?

Suppose our normalized scores became:
```text
[0.18, 0.14, 0.10, 0.07, 0.16, 0.09, 0.26]
```

Now we can interpret them much more easily.

For example:
```text
The       -> 18%
bank      -> 14%
said      -> 10%
there     ->  7%
was       -> 16%
no        ->  9%
network   -> 26%
```

The values add up to:

$ 1.0 $

So we can think of them as describing **how the attention is distributed across the tokens**.

But how do we convert arbitrary scores into this kind of distribution?

We use **softmax**.

##### **Understanding Softmax**

##### The Idea Behind Softmax

Softmax takes a collection of numers and converts them into values between 0 and 1 whose sum is 1.

Conceptually:

     scores

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

    softmax

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

    attention weights

One important property of softmax is that it preserves the **relative ordering** of the scores.

If one token has a higher score than another token, it will receive a higher attention weight.

So:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ score_A > score_B $

generally means:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ weight_A > weight_B $

Softmax also makes the differences between scores matter more clearly.

##### How Does Softmax Actually Work?

For a vector of scores:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ [x_1,x_2,...,x_n] $

softmax calculates each output as:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ softmax(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}} $


**1. Exponentiation**

Each score is passed through the exponential function:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ e^{x_i} $

This makes larger scores grow faster than smaller scores.

**2. Normalization**

Each exponential value is divided by the sum of all exponential values:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \frac{e^{x_i}}{\sum_j e^{x_j}} $

This makes all the resulting values add up to **1**.

So softmax is doing both:
> **Turn scores into positive values -> normalize them into a distribution.**

##### Let's See This With Simple Numbers

Before applying softmax to our attention scores, let's use a very small example.

Suppose we have:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ [1,2,3] $

First, calculate the exponentials:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ e^1,e^2,e^3 $

Then divide each one by their total:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \frac{e^1}{e^1 + e^2 + e^3} $

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \frac{e^2}{e^1 + e^2 + e^3} $

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \frac{e^3}{e^1 + e^2 + e^3} $

Let's use PyTorch to calculate the exponential values first.

##### Exponentials

In [13]:
scores = torch.tensor([1.0, 2.0, 3.0])
exp_scores = torch.exp(scores)
print(exp_scores)

tensor([ 2.7183,  7.3891, 20.0855])


Notice what happened.

The difference between the original scores was:
```text
1, 2, 3
```

But after exponentiation:
```text
2.72, 7.39, 20.09
```

The larger score has become much more dominant.

##### Normalize the Exponential Values

Now we divide every exponential value by the sum of all exponential values.

In [14]:
sum_exp_scores = exp_scores.sum()
attention_weights = exp_scores / sum_exp_scores
print(attention_weights)

tensor([0.0900, 0.2447, 0.6652])


In [15]:
print(attention_weights.sum())

tensor(1.)


And every value is between 0 and 1.

We've just implemented the basic idea of softmax ourselves.

##### Now Let's Use PyTorch's `softmax()`

Just like we did with the dot product, we first understood the operation manually.

Now we can use PyTorch's built-in implementation.
```python
torch.softmax()
```

In [16]:
attention_weights = torch.softmax(scores, dim=0)
print(attention_weights)

tensor([0.0900, 0.2447, 0.6652])


This should produce the same values as our manual calculation.

We can verify:

In [17]:
manual = exp_scores / exp_scores.sum()
pytorch = torch.softmax(scores, dim=0)
print(torch.allclose(manual, pytorch))

True


So:
> **`torch.softmax()` is not doing something mysterious. It is implementing the mathematical softmax operation we just built ourselves.**

##### **Apply Softmax to Our Attention Scores**

##### Normalize Our Attention Scores

Now let's return to the actual attention scores for `"bank"`.

We currently have:
```python
attention_scores
```

These are our raw scores.

We'll apply softmax to them.

Because each element in `attention_scores` corresponds to a token, we want softmax to operate across that entire vector.

In [18]:
attention_weights = torch.softmax(attention_scores, dim=0)
print(attention_weights)

tensor([0.1330, 0.1742, 0.1513, 0.1235, 0.1503, 0.1241, 0.1437])


##### Look at the Attention Distribution

Let's associate each weight with its token.

In [19]:
print("Token    attention score    attention weight")
print("-" * 50)
for token, score, weight in zip(tokens, attention_scores, attention_weights):
    print(f"{token:8}\t{score:.4f}\t\t{weight:.4f}")

Token    attention score    attention weight
--------------------------------------------------
The     	0.8498		0.1330
bank    	1.1198		0.1742
said    	0.9790		0.1513
there   	0.7757		0.1235
was     	0.9727		0.1503
no      	0.7808		0.1241
network 	0.9277		0.1437


Now we can see where the "bank" query is distributing its attention.

The weights should satisfy two important properties:

1. Every weight is between 0 and 1.
2. All weights add up to 1.
   
Let's verify both.

In [20]:
print("Sum of attention weights:", attention_weights.sum())
print("Smallest weight:\t ", attention_weights.min())
print("Largest weight:\t\t ", attention_weights.max())

Sum of attention weights: tensor(1.)
Smallest weight:	  tensor(0.1235)
Largest weight:		  tensor(0.1742)


##### What Have We Built?

We have gone from:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \boxed{\text{Token representations}} $

to:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \boxed {\text{Dot products}} $

to:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \boxed {\text{Attention scores}} $

to:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \boxed {\text{Attention weights}} $

The process so far is:

    Token representations

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

    Compare vectors

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

    Dot product

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

    Raw attention scores

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

    Softmax

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

    Attention weights

But we still haven't accomplished the main goal of attention.

We have calculated **how much attention to give each token**, but we haven't yet used those weights to **actually gather information from the tokens.**

That's the next step:
> **Use the attention weights to create a context vector.**

#### 3.2.4: From Attention Weights to Context Vectors

##### **What Are We Trying to Build?**

Our attention weights tell us **how much attention the query should give to each token**.

For `"bank"`, we now have something conceptually like:
```text
The       -> weight
bank      -> weight
said      -> weight
there     -> weight
was       -> weight
no        -> weight
network   -> weight
```

But we want to turn these weights into a new vector.

This new vector should contain information gathered from the entire sequence, weighted according to how much attention `"bank"` gives each token.

We call this a **context vector**.

##### **The Basic Idea**

Suppose our attention weights were:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ [0.10,\ 0.20,\ 0.05,\ 0.10,\ 0.15,\ 0.10,\ 0.30] $

and we had seven token representations:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ x_1, x_2,..., x_7 $

We calculate the context vector as:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ z = w_1x_1 + w_2x_2 + \cdots + w_7x_7 $

In other words:
> **Multiply each token representation by its attention weight, then add all the weighted vectors together.**

The result is one new vector.

##### **Why Weight the Vectors?**

Think about what the weights are telling us.

If a token receives a high attention weight:
```text
attention weight = 0.30
```
then its representation contributes **more** to the context vector.

If another token receives a low weight:
```text
attention weight = 0.05
```
then its representation contributes **less**.

So attention is effectively saying:
> “Take information from all these tokens, but don't treat them equally.”

##### **From One Query to Every Token**

So far, we have focused on only one query:
> **“bank”**

We compared `"bank"` with every token and obtained one row of attention weights:
```text
bank -> [weight for The, bank, said, there, was, no, network]
```

But `"bank"` is not the only token that needs contextual information.

Every token should be able to look at the other tokens.

So we need to repeat the same process for:
```text
The
bank
said
there
was
no
network
```

Each token becomes a query.

Therefore, instead of having one row of attention weights, we will eventually have seven rows.

##### **What This Looks Like**

Conceptually:
```text
             The   bank  said  there  was   no   network

The          ?      ?     ?      ?     ?     ?      ?

bank         ?      ?     ?      ?     ?     ?      ?

said         ?      ?     ?      ?     ?     ?      ?

there        ?      ?     ?      ?     ?     ?      ?

was          ?      ?     ?      ?     ?     ?      ?

no           ?      ?     ?      ?     ?     ?      ?

network      ?      ?     ?      ?     ?     ?      ?
```

Each **row** represents one query.

Each **column** represents the token being attended to.

Our `"bank"` attention weights are just the **second row**.

##### **Calculate Attention Weights for Every Query**

In [21]:
# Create an empty matrix to store the attention weights.
# There is one row for each query token and one column for each token being attended to.
attention_weights = torch.empty(inputs.shape[0], inputs.shape[0])

# Treat each token representation as a query, one at a time.
for i, query in enumerate(inputs):

    # Create an empty vector to store the attention scores
    # between the current query and every token in the sequence.
    attention_scores = torch.empty(inputs.shape[0])

    # Compare the current query with every other tokken representation.
    for j, x_j in enumerate(inputs):

        # Calculate the dot product between the query and the current token.
        # This gives us the raw attention score for the pair of tokens.
        attention_scores[j] = torch.dot(query, x_j)

    # Convert the raw attention scores into attention weights.
    # The weights for this query will sum to 1.
    attention_weights[i] = torch.softmax(attention_scores, dim=0)

print("Attention weights for every query:\n", attention_weights)

Attention weights for every query:
 tensor([[0.1550, 0.1651, 0.1330, 0.1439, 0.1349, 0.1314, 0.1367],
        [0.1330, 0.1742, 0.1513, 0.1235, 0.1503, 0.1241, 0.1437],
        [0.1008, 0.1424, 0.1959, 0.0992, 0.1544, 0.1497, 0.1576],
        [0.1509, 0.1608, 0.1372, 0.1415, 0.1360, 0.1350, 0.1386],
        [0.1148, 0.1590, 0.1734, 0.1104, 0.1566, 0.1341, 0.1516],
        [0.1175, 0.1378, 0.1766, 0.1151, 0.1409, 0.1599, 0.1523],
        [0.1154, 0.1508, 0.1756, 0.1116, 0.1504, 0.1439, 0.1522]])


This can be done more efficiently using matrix multiplication:

In [22]:
# Compute attention scores
attention_scores = inputs @ inputs.T
print("Attention scores (matrix multiplication method):\n", attention_scores)

attention_weights = torch.softmax(attention_scores, dim=-1)
print("\nAttention weights (from multiplication method done for attention scores):\n", attention_weights)

Attention scores (matrix multiplication method):
 tensor([[0.7864, 0.8498, 0.6336, 0.7118, 0.6472, 0.6212, 0.6606],
        [0.8498, 1.1198, 0.9790, 0.7757, 0.9727, 0.7808, 0.9277],
        [0.6336, 0.9790, 1.2979, 0.6169, 1.0595, 1.0285, 1.0804],
        [0.7118, 0.7757, 0.6169, 0.6477, 0.6078, 0.6007, 0.6272],
        [0.6472, 0.9727, 1.0595, 0.6078, 0.9573, 0.8027, 0.9253],
        [0.6212, 0.7808, 1.0285, 0.6007, 0.8027, 0.9291, 0.8808],
        [0.6606, 0.9277, 1.0804, 0.6272, 0.9253, 0.8808, 0.9374]])

Attention weights (from multiplication method done for attention scores):
 tensor([[0.1550, 0.1651, 0.1330, 0.1439, 0.1349, 0.1314, 0.1367],
        [0.1330, 0.1742, 0.1513, 0.1235, 0.1503, 0.1241, 0.1437],
        [0.1008, 0.1424, 0.1959, 0.0992, 0.1544, 0.1497, 0.1576],
        [0.1509, 0.1608, 0.1372, 0.1415, 0.1360, 0.1350, 0.1386],
        [0.1148, 0.1590, 0.1734, 0.1104, 0.1566, 0.1341, 0.1516],
        [0.1175, 0.1378, 0.1766, 0.1151, 0.1409, 0.1599, 0.1523],
        [0.1154

Now the shape should be:

In [23]:
print("Attention weights shape:\n", attention_weights.shape)

Attention weights shape:
 torch.Size([7, 7])


So we have:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ 7 \text{ queries} \times 7 \text{ tokens} $

##### **Inspect the Attention Weights**

Let's make the matrix easier to understand by printing the tokens alongside it.

In [24]:
print("           " + " ".join(f"{token:>8}" for token in tokens))

for token, weights in zip(tokens, attention_weights):
    values = "   ".join(f"{weight:.4f}" for weight in weights)
    print(f"{token:>8}     {values}")

                The     bank     said    there      was       no  network
     The     0.1550   0.1651   0.1330   0.1439   0.1349   0.1314   0.1367
    bank     0.1330   0.1742   0.1513   0.1235   0.1503   0.1241   0.1437
    said     0.1008   0.1424   0.1959   0.0992   0.1544   0.1497   0.1576
   there     0.1509   0.1608   0.1372   0.1415   0.1360   0.1350   0.1386
     was     0.1148   0.1590   0.1734   0.1104   0.1566   0.1341   0.1516
      no     0.1175   0.1378   0.1766   0.1151   0.1409   0.1599   0.1523
 network     0.1154   0.1508   0.1756   0.1116   0.1504   0.1439   0.1522


##### **Verify Every Row**

In [25]:
row_sums = attention_weights.sum(dim=1)
print(row_sums)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


This is important because **softmax was applied independently to each query's scores**.

##### **What Have We Got?**

We now have:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;  $ A = 
\begin{bmatrix}
\text{weights for The} \\
\text{weights for bank} \\
\text{weights for said} \\
\text{weights for there} \\
\text{weights for was} \\
\text{weights for no} \\
\text{weights for network} \\
\end{bmatrix}
$

Every token has its own attention distribution.

For example:
```text
The      -> where should "The" pay attention?
bank     -> where should "bank" pay attention?
said     -> where should "said" pay attention?
there    -> where should "there" pay attention?
was      -> where should "was" pay attention?
no       -> where should "no" pay attention?
network  -> where should "network" pay attention?
```

This is much closer to what we mean by **self-attention**.

Every token is allowed to look at the other tokens and determine how much attention to give each one.

##### **Now We Can Build All Context Vectors**

At this point, **then** we move to the weighted-sum operation.

Instead of calculating only:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;  $ z_{\text{bank}} = \sum_jA_{\text{bank},j}x_j $

we calculate it for every **token**:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;  $ z_i = \sum_jA_{ij}x_j $

So every row of the attention-weight matrix produces one context vector.

That gives us:

7 attention distributions

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

7 weighted sums

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

7 context vectors

And that is the point where the matrix operation:
```python
context_vectors = attention_weights @ inputs
```

##### **Let's Calculate One Contribution**

Let's take the token `"network"`.

Suppose its attention weight for our `"bank"` query is:
```python
attention_weights[6]
```
Its representation is:
```python
inputs[6]
```

Its contribution to the context vector is:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ attention\_weight_{network} \times network_\_representation $

Let's calculate that explicitly.

In [26]:
network_weight = attention_weights[1][6] # Attention weight for 'bank' query
network_vector = inputs[6] # Token 'network' representation

network_contribution = network_weight * network_vector

print("Network weight:")
print(network_weight)

print("\nNetwork representation:")
print(network_vector)

print("\nNetwork contribution:")
print(network_contribution)

Network weight:
tensor(0.1437)

Network representation:
tensor([0.5800, 0.6700, 0.3900])

Network contribution:
tensor([0.0834, 0.0963, 0.0561])


Notice that the result is still a **3-dimensional vector**.

We have taken the original `"network"` representation and scaled it according to how much attention `"bank"` gives it.

##### **Do This for Every Token**

We need to perform the same operation for every token:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ w_1x_1 $

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ w_2x_2 $

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ w_3x_3 $

and so on.

Then we add all of those vectors together:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ z = w_1x_1 + w_2x_2 + \cdots + w_7x_7 $

Let's first implement this explicitly with a loop.

##### **Build the Context Vector Manually**

In [27]:
context_vector = torch.zeros(inputs.shape[1])

print("Token\t\tattention weight\tToken vector attended too by input token 'bank'")
print("-" * 100)
for i, x_i in enumerate(inputs):
    print(f"{tokens[i]}\t\t{attention_weights[1][i]:.4f}\t\t\t{x_i}")
    context_vector += attention_weights[1][i] * x_i

print("\nContext vector for input token 'bank':\n", context_vector)

Token		attention weight	Token vector attended too by input token 'bank'
----------------------------------------------------------------------------------------------------
The		0.1330			tensor([0.4200, 0.1800, 0.7600])
bank		0.1742			tensor([0.8100, 0.3400, 0.5900])
said		0.1513			tensor([0.6300, 0.9100, 0.2700])
there		0.1235			tensor([0.3700, 0.2200, 0.6800])
was		0.1503			tensor([0.7400, 0.5600, 0.3100])
no		0.1241			tensor([0.2900, 0.7900, 0.4700])
network		0.1437			tensor([0.5800, 0.6700, 0.3900])

Context vector for input token 'bank':
 tensor([0.5685, 0.5265, 0.4896])


##### **Let's See Each Contribution**

It can be useful to see what is happening before we simply look at the final result.

In [28]:
for token, weight, x_i in zip(tokens, attention_weights, inputs):
    contribution = weight[1] * x_i

    print(f"{token:8} weight = {weight[1]:.4f}")
    print(f"         contribution = {contribution}\n")

The      weight = 0.1651
         contribution = tensor([0.0694, 0.0297, 0.1255])

bank     weight = 0.1742
         contribution = tensor([0.1411, 0.0592, 0.1028])

said     weight = 0.1424
         contribution = tensor([0.0897, 0.1296, 0.0385])

there    weight = 0.1608
         contribution = tensor([0.0595, 0.0354, 0.1094])

was      weight = 0.1590
         contribution = tensor([0.1177, 0.0890, 0.0493])

no       weight = 0.1378
         contribution = tensor([0.0400, 0.1089, 0.0648])

network  weight = 0.1508
         contribution = tensor([0.0874, 0.1010, 0.0588])



Each token contributes a vector.

The attention weight determines **how much of that vector gets included**.

##### **The Context Vector**

After adding all seven weighted vectors together, we obtain:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ z = w_1x_1 + w_2x_2 + \cdots + w_7x_7 $

This is the **context vector for `"bank"`**.

It is no longer simply the original representation of `"bank"`.

Instead, it incorporates information from the entire sequence according to the attention distribution.

Let's inspect it alongside the original representation.

In [29]:
print("Original representation of 'bank':\n", inputs[1])
print("\nContext vector for 'bank':\n", context_vector)

Original representation of 'bank':
 tensor([0.8100, 0.3400, 0.5900])

Context vector for 'bank':
 tensor([0.5685, 0.5265, 0.4896])


- We can compute context vectors for every token using matrix multiplication

In [30]:
context_vectors = attention_weights @ inputs
print("Context vectors for every token:\n", context_vectors)

Context vectors for every token:
 tensor([[0.5531, 0.5077, 0.5059],
        [0.5685, 0.5265, 0.4896],
        [0.5669, 0.5770, 0.4607],
        [0.5526, 0.5135, 0.5025],
        [0.5698, 0.5521, 0.4737],
        [0.5537, 0.5612, 0.4747],
        [0.5639, 0.5563, 0.4736]])


##### **What Just Happened?**

This is the core idea behind attention.

Originally:

  bank
    
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

[0.81, 0.34, 0.59]


The representation of `"bank"` was based only on its original vector.

After attention:

  bank

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

  looks at all tokens
    
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

  calculates attention weights
    
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

  collects information from all tokens
    
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

  context vector

So the representation has become **contextualized**.

The new vector reflects information gathered from the surrounding sequence.

- Check context vector for input token `"bank"` computed before is the same as the context vector computed using matrix multiplication which is found in the 2nd row:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ [0.5685, 0.5265, 0.4896] $

In [31]:
print("Context vector of input token 'bank' computed with matrix multiplication:\n", context_vectors[1]) # Python is 0-indexed

Context vector of input token 'bank' computed with matrix multiplication:
 tensor([0.5685, 0.5265, 0.4896])


##### **The Entire Process So Far**

We can now see the complete simple attention mechanism:

Token representations

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

Choose a query

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;
        
Compare query with tokens

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

Dot products

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

Attention scores

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

Softmax

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

Attention weights

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

Weighted token representations

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

Sum

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;
        
Context vector


Mathematically, our simplified mechanism is:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ S = XX^T $

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ A = softmax(S) $

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ Z = AX $

where:
- $ X $ = token representations
- $ S $ = attention scores
- $ A $ = attention weights
- $ Z $ = context vectors

But notice something important.

We have been using the **same token representations** for all three roles:

- the thing asking the question,
- the thing being compared against,
- and the information being retrieved.
  
In other words, we've effectively assumed:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ Q = X $

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ K = X $

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ V = X $

This was useful for understanding the basic mechanism.

But real attention doesn't normally make this assumption.

And that brings us to the next major idea:

##### **Query, Key, and Value**
That's where attention becomes a **learned mechanism**, rather than simply comparing the original token representations directly.

## Part 2

### 3.3 Query, Key, and Value

##### **Why Do We Need Query, Key, and Value?**

Our simple attention mechanism currently does this:

Token representations

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

compare

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;
        
Attention scores

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

Attention weights

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

Weighted token representations

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

Context vectors

The problem is that we are using the **same representation for every role**.

For example, when `"bank"` looks at `"network"`, we are essentially doing:

"bank" representation  -> query

"network" representation -> key

"network" representation -> value

using the original token representations directly.

But these three roles are actually different.

Attention needs to answer three different questions:

**1. What am I looking for?** -> Query

**2. What do I contain that can be matched?** -> Key

**3. What information should I provide if I am selected?** -> Value

This is why attention introduces **Query, Key, and Value**.

##### An Intuition

Imagine `"bank"` is trying to find useful information from the other tokens.

The `"bank"` token produces a **Query**:

> “What kind of information am I looking for?”

Every token produces a **Key**:

> “What kind of information can I be matched on?”

And every token also produces a **Value**:

> “What information should I provide if you attend to me?”

So attention works roughly like this:

                 Query
                  ↓
              "What am I
               looking for?"
                  │
                  ↓
        ┌─────────────────────┐
        │ Compare with Keys   │
        └─────────────────────┘
          ↓      ↓      ↓
        Key    Key    Key ...
          │      │      │
          └──────┴──────┘
                  ↓
          Attention scores
                  ↓
          Attention weights
                  ↓
        Use weights on Values
                  ↓
          Context vector

The important distinction is:
> **Queries and Keys determine where to pay attention. Values provide the information that gets gathered.**

##### **Query, Key, and Value for Our Sentence**

For:

> **“The bank said there was no network.”**

every token gets its own Query, Key, and Value representation.

For example, `"bank"` has:

```text
bank
 ├── Query → what "bank" is looking for
 ├── Key   → what "bank" can be matched on
 └── Value → information "bank" can provide
```

And `"network"` has:

```text
network
 ├── Query → what "network" is looking for
 ├── Key   → what "network" can be matched on
 └── Value → information "network" can provide
```

These aren't three different tokens.

They are **three different learned representations of the same token**.

##### **How Are Q, K, and V Created?**

We don't manually decide what the Query, Key, and Value should contain.

The model **learns them**.

Starting with our input matrix:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ X $

we create three new representations using three trainable weight matrices:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ Q = XW_Q $

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ Q = XW_K $

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ Q = XW_V $

where:
- $ X $ = input token representations
- $ W_Q $ = Query projection weights
- $ W_K $ = Key projection weights
- $ W_Q $ = Value projection weights

The important idea is that **the same input representation is projected into three different spaces.**

##### **Why Three Different Projections?**

If we used the same representation for everything, we would force one vector to perform three different jobs.

With separate projections, the model can learn:
```text
X
│
├── WQ → Query representation
│
├── WK → Key representation
│
└── WV → Value representation
```

During training, the model learns the values of $ W_Q $, $ W_K $, and $ W_V $ so that these representations become useful for attention.

This is what makes the attention mechanism **learnable**.

##### **Our Attention Process Is Changing**

Previously, we had:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ S = XX^T $

because we were using $ X $ directly as both the query and key.

Now we will have:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ S = QK^T $

The attention scores are calculated by comparing **Queries with Keys**.

Then:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ A = softmax(S) $

And instead of using the original input representations directly, we use the **Values**:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ Z = AV $

So the complete simplified process becomes:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ X→Q,K,V $

then:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ QK^T→\text{attention scores} $

then:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ softmax→\text{attention weights} $

then:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ AV→\text{context vectors} $

This is the key conceptual upgrade from our simple attention mechanism.

##### **Let's Create the Projection Matrices**

For now, we'll use small randomly initialized matrices so we can see the mechanics.

Later, we'll discuss why these weights are trainable and how they are learned through backpropagation.

Our input vectors have 3 dimensions, so we'll initially keep the Query, Key, and Value dimensions at 3 as well.

In [32]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(3, 3))
print("Query weight parameters:\n", W_query)

W_key = torch.nn.Parameter(torch.rand(3, 3))
print("\nKey weight parameters:\n", W_key)

W_value = torch.nn.Parameter(torch.rand(3, 3))
print("\nValue weight parameters:\n", W_value)

Query weight parameters:
 Parameter containing:
tensor([[0.2961, 0.5166, 0.2517],
        [0.6886, 0.0740, 0.8665],
        [0.1366, 0.1025, 0.1841]], requires_grad=True)

Key weight parameters:
 Parameter containing:
tensor([[0.7264, 0.3153, 0.6871],
        [0.0756, 0.1966, 0.3164],
        [0.4017, 0.1186, 0.8274]], requires_grad=True)

Value weight parameters:
 Parameter containing:
tensor([[0.3821, 0.6605, 0.8536],
        [0.5932, 0.6367, 0.9826],
        [0.2745, 0.6584, 0.2775]], requires_grad=True)


##### **Generate Queries, Keys, and Values**

Now we project our input representations into the three different spaces.

In [33]:
queries = inputs @ W_query
keys = inputs @ W_key
values = inputs @ W_value

print("Queries shape:", queries.shape)
print("Keys shape:", keys.shape)
print("Values shape:", values.shape)

Queries shape: torch.Size([7, 3])
Keys shape: torch.Size([7, 3])
Values shape: torch.Size([7, 3])


We still have seven tokens.

But each token now has **three different representations**.

##### **Look at One Token**

Let's inspect `"bank"`.

In [34]:
bank_index = 1

print("Token:", tokens[bank_index])
print("\nQuery:\n", queries[bank_index])
print("\nKey:\n", queries[bank_index])
print("\nValue:\n", queries[bank_index])

Token: bank

Query:
 tensor([0.5545, 0.5040, 0.6071], grad_fn=<SelectBackward0>)

Key:
 tensor([0.5545, 0.5040, 0.6071], grad_fn=<SelectBackward0>)

Value:
 tensor([0.5545, 0.5040, 0.6071], grad_fn=<SelectBackward0>)


The important thing to notice is that these three vectors are different.

They all came from the same `"bank"` input representation, but each was produced using a different learned projection matrix.

##### **How Attention Uses Them**

Now we can finally make the roles precise.

Suppose `"bank"` is the query.

We compare its Query against every token's Key:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ q_{bank} \cdot k_{The} $

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ q_{bank} \cdot k_{bank} $

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ q_{bank} \cdot k_{said} $

and so on.

These comparisons produce the attention scores.

Then we apply softmax:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ softmax(QK^T) $

to obtain the attention weights.

Finally, those weights are applied to the **Values**:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ AV $

So:
> **Queries and Keys decide where attention goes. Values determine what information is actually gathered.**

That distinction is one of the most important ideas in understanding modern attention.

##### **Calculate One Query–Key Score**

Let's focus on `"bank"` again.

Its Query is:
```python
print(queries[1])
```
Now let's compare it with the Key of `"network"`:
```python
print(keys[6])
```

The attention score is:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ q_{bank} \cdot k_{network} $

Notice that we are **not using the Value yet**.

The Value will only be used after we have calculated the attention weights.

##### **Inspect the Query and Key**

In [35]:
bank_query = queries[1]
network_key = keys[6]

print("Bank Query:\n", bank_query)
print("\nNetwork Key:\n", network_key)

Bank Query:
 tensor([0.5545, 0.5040, 0.6071], grad_fn=<SelectBackward0>)

Network Key:
 tensor([0.6287, 0.3608, 0.9332], grad_fn=<SelectBackward0>)


##### **Calculate the Dot Product Manually**

Just as we did earlier, let's avoid hiding the operation behind PyTorch.

The dot product is::

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ q_{bank} \cdot k_{network} $

which means:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ (q_1k_1) + (q_2k_2) + (q_3k_3) $

Let's calculate each component.

In [36]:
part_1 = bank_query[0] * network_key[0]
print("First dimension:\n", part_1)

part_2 = bank_query[1] * network_key[1]
print("\nSecond dimension:\n", part_2)

part_3 = bank_query[2] * network_key[2]
print("\nThird dimension:\n", part_3)

bank_network_score = part_1 + part_2 + part_3
print("\nAttention score (using manual method):", bank_network_score)

First dimension:
 tensor(0.3486, grad_fn=<MulBackward0>)

Second dimension:
 tensor(0.1819, grad_fn=<MulBackward0>)

Third dimension:
 tensor(0.5665, grad_fn=<MulBackward0>)

Attention score (using manual method): tensor(1.0970, grad_fn=<AddBackward0>)


##### **Using `torch.dot()`**

Now that we've seen what happens internally, we can use PyTorch's implementation:

In [37]:
bank_network_score = torch.dot(bank_query, network_key)
print("Attention score (using dot method):", bank_network_score)

Attention score (using dot method): tensor(1.0970, grad_fn=<DotBackward0>)


So our attention score is now based on:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \boxed{\text{QK}} $

rather than:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \boxed{\text{XX}} $

That's the fundamental change introduced by Query and Key projections.

##### **From One Score to All Scores**

##### **Compare One Query With Every Key**

We don't just want to know how `"bank"` relates to `"network"`.

We want to compare the Query for `"bank"` with the Key of **every token**:
```text
bank Query
    ↓
    ├── The Key
    ├── bank Key
    ├── said Key
    ├── there Key
    ├── was Key
    ├── no Key
    └── network Key
```

Each comparison produces one attention score.

In [38]:
bank_query = queries[1]

# Create a tensor filled with unintialized data to store the attention scores.
# torch.empty allocates a block of memory of the specified size
attention_scores = torch.empty(inputs.shape[0])

for i, key in enumerate(keys):
    # Calculate the dot product between the 'bank' query and each key.
    # This gives us the raw attention score for each 'bank' query and each key.
    attention_scores[i] = torch.dot(bank_query, key)

print("Attention scores for 'bank' and each key:\n", attention_scores)

Attention scores for 'bank' and each key:
 tensor([1.0676, 1.3692, 1.1317, 0.9691, 1.1542, 0.9159, 1.0970],
       grad_fn=<CopySlices>)


Now we have the raw attention scores for `"bank"` using **Query–Key comparisons**.

##### **Label the Scores**

Let's see which score belongs to which token.

In [39]:
print("Token    Attention score")
for token, score in zip(tokens, attention_scores):
    print(f"{token:8} {score:.4f}")

Token    Attention score
The      1.0676
bank     1.3692
said     1.1317
there    0.9691
was      1.1542
no       0.9159
network  1.0970


Conceptually:
```text
bank Query
     ↓
     ├── The      → score
     ├── bank     → score
     ├── said     → score
     ├── there    → score
     ├── was      → score
     ├── no       → score
     └── network  → score
```

These are still **raw attention scores**.

We haven't normalized them yet.

#### 3.3.1 Calculate the Scores for Every Query

##### **Generalizing to the Whole Sequence**

Just as before, we don't want attention weights only for `"bank"`.

Every token needs its own Query.

Each Query is compared with every Key.

Therefore, we need:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ 7 \times 7 $

Query–Key comparisons.

That gives us a **7 × 7 attention-score matrix**.

In [40]:
# Create a tensor filled with unintialized data to store the attention scores.
# torch.empty allocates a block of memory of the specified size
attention_scores = torch.empty(inputs.shape[0], inputs.shape[0])

# Treat each token representation as a query, one at a time.
for i, query in enumerate(queries):
    
    # Compare the current query with every key representation.
    for j, key in enumerate(keys):

        # Calculate the dot product between the the current query and each key.
        # This gives us the raw attention score for the current query and each key.
        attention_scores[i, j] = torch.dot(query, key)

print("Attention scores for every query:\n", attention_scores)

Attention scores for every query:
 tensor([[0.6905, 0.8833, 0.7289, 0.6268, 0.7435, 0.5915, 0.7073],
        [1.0676, 1.3692, 1.1317, 0.9691, 1.1542, 0.9159, 1.0970],
        [1.6101, 2.0370, 1.6531, 1.4596, 1.6977, 1.3529, 1.6163],
        [0.6908, 0.8811, 0.7243, 0.6268, 0.7399, 0.5891, 0.7041],
        [1.2311, 1.5689, 1.2853, 1.1168, 1.3154, 1.0455, 1.2510],
        [1.3216, 1.6639, 1.3426, 1.1976, 1.3816, 1.1037, 1.3165],
        [1.3065, 1.6567, 1.3490, 1.1847, 1.3836, 1.1020, 1.3169]],
       grad_fn=<CopySlices>)


Now:

- each **row** represents a Query
- each **column** represents a Key

So:
```text
                  Keys
             The bank said there was no network

Query  The      ·    ·    ·     ·    ·    ·     ·
       bank     ·    ·    ·     ·    ·    ·     ·
       said     ·    ·    ·     ·    ·    ·     ·
       there    ·    ·    ·     ·    ·    ·     ·
       was      ·    ·    ·     ·    ·    ·     ·
       no       ·    ·    ·     ·    ·    ·     ·
       network  ·    ·    ·     ·    ·    ·     ·
```

This is the **Query-Key attention-score matrix.**

#### 3.3.2 Where Does the Value Come In?

##### **We Have Not Used Values Yet**

Notice something important.

So far, we've used:

Query -> Key (Query attending to the keys)

to determine the attention scores.

We then use those scores to determine the attention weights.

But we haven't used the Values.

That's intentional.

The three components have different jobs:
```text
Query + Key
      ↓
Determine attention
      ↓
Attention weights
      ↓
Applied to Values
      ↓
Context vectors
```

So the Value does **not** participate in calculating the attention score.

It provides the information that gets aggregated **after** we've decided where to pay attention.

##### **Normalize the Query–Key Scores**

Now we can apply softmax to each row of our new attention-score matrix.

Each Query gets its own attention distribution over all Keys.

In [41]:
attention_weights = torch.softmax(attention_scores, dim=-1)
print("Attention weights:\n", attention_weights)

Attention weights:
 tensor([[0.1395, 0.1692, 0.1450, 0.1309, 0.1471, 0.1264, 0.1419],
        [0.1369, 0.1851, 0.1460, 0.1241, 0.1493, 0.1176, 0.1410],
        [0.1368, 0.2097, 0.1429, 0.1177, 0.1494, 0.1058, 0.1377],
        [0.1399, 0.1692, 0.1447, 0.1312, 0.1469, 0.1264, 0.1418],
        [0.1372, 0.1923, 0.1448, 0.1224, 0.1493, 0.1140, 0.1400],
        [0.1394, 0.1963, 0.1424, 0.1231, 0.1480, 0.1121, 0.1387],
        [0.1379, 0.1957, 0.1438, 0.1220, 0.1489, 0.1124, 0.1393]],
       grad_fn=<SoftmaxBackward0>)


Let's verify that every row sums to 1.

In [42]:
print(attention_weights.sum(dim=-1))

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       grad_fn=<SumBackward1>)


#### 3.3.3 Use the Values

##### **Create the Context Vectors**

Now we finally use the third component of attention:

**Values.**

For each query, we take the attention weights and use them to calculate a weighted combination of the Value vectors.

For one query:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ z_i = \sum\limits_{j}a_{ij}v_j $

where:
- $ a_{ij} $ = attention weight
- $ v_j $ = Value of token $ j $
- $ z_i $ = resulting context vector

So our complete process is now:

&nbsp;&nbsp;&nbsp;&nbsp; $ Q,K,V $
    
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ QK^T $

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

$ \text{attention scores} $

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

&nbsp;&nbsp;$ softmax $

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

$ \text{attention scores} $

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ AV $

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&darr;

$ \text{context vectors} $

##### **Calculate Context Vectors Explicitly**

Let's first do this with loops, so the mathematics remains visible.

In [43]:
# Create an uninitialized tensor to store the context vector for each token.
# It has the same shape as the Value matrix:
# one context vector for each token.
context_vectors = torch.empty_like(values)

# Process each row of attention weights.
# Each row represents the attention distribution for one query token.
for i, weights in enumerate(attention_weights):

    # Create an empty vector to accumulate the weighted
    # Value vectors for the current query.
    context_vector = torch.zeros(values.shape[1])

    # Go through every Value vector in the sequence.
    for j, value in enumerate(values):

        # Multiply the Value vector by its attention weight
        # and add the result to the context vector.
        print(weights[j], value)
        context_vector += weights[j] * value

    # Store the completed context vector for the current query.
    context_vectors[i] = context_vector
    print()

print("Context vectors:\n", context_vectors)

tensor(0.1395, grad_fn=<SelectBackward0>) tensor([0.4759, 0.8924, 0.7463], grad_fn=<UnbindBackward0>)
tensor(0.1692, grad_fn=<SelectBackward0>) tensor([0.6731, 1.1399, 1.1892], grad_fn=<UnbindBackward0>)
tensor(0.1450, grad_fn=<SelectBackward0>) tensor([0.8546, 1.1733, 1.5069], grad_fn=<UnbindBackward0>)
tensor(0.1309, grad_fn=<SelectBackward0>) tensor([0.4585, 0.8322, 0.7207], grad_fn=<UnbindBackward0>)
tensor(0.1471, grad_fn=<SelectBackward0>) tensor([0.7000, 1.0494, 1.2680], grad_fn=<UnbindBackward0>)
tensor(0.1264, grad_fn=<SelectBackward0>) tensor([0.7084, 1.0040, 1.1543], grad_fn=<UnbindBackward0>)
tensor(0.1419, grad_fn=<SelectBackward0>) tensor([0.7261, 1.0665, 1.2617], grad_fn=<UnbindBackward0>)

tensor(0.1369, grad_fn=<SelectBackward0>) tensor([0.4759, 0.8924, 0.7463], grad_fn=<UnbindBackward0>)
tensor(0.1851, grad_fn=<SelectBackward0>) tensor([0.6731, 1.1399, 1.1892], grad_fn=<UnbindBackward0>)
tensor(0.1460, grad_fn=<SelectBackward0>) tensor([0.8546, 1.1733, 1.5069], grad_f

Now every token has its own contextualized representation.

##### **The Important Change**

Previously, our simplified attention used:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ Z = AX $

because the original token representations were also acting as Values.

Now we have:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \boxed{Z = AV} $

because the model has learned a separate Value representation.

And our attention scores have changed from:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ XX^T $

to:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \boxed{QK^T} $

So we have moved from our simplified attention mechanism to the structure used in actual Transformer attention.

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \boxed{Attention(Q,K,V) = softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V}$


### 3.4 Scaling the Attention Scores

So far, we have calculated our attention scores using the dot product between Queries and Keys.

For example:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ score(Q,K) = Q \cdot K $

We then pass these scores through softmax to turn them into attention weights.

However, there is a problem.

As the dimensionality of our Query and Key vectors increases, their dot products can become larger in magnitude.

Large scores can cause softmax to become very sharp. A few tokens may receive almost all of the attention, while the remaining tokens receive values very close to zero.

This can make learning difficult because the gradients produced by softmax can become very small.

To reduce this problem, we scale the attention scores before applying softmax.

The standard scaled dot-product attention equation is:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V $

The important part for now is:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \frac{QK^T}{\sqrt{d_k}} $

where $ d_k $ is the dimensionality of the Key vectors.

In our example, each Key has 3 dimensions, so:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ d_k = 3 $

and therefore:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \sqrt{d_k} = \sqrt{3} $

Scaling does **not** normalize the attention weights.

The division by $ \sqrt{d_k} $ simply reduces the magnitude of the raw attention scores.

Softmax is still responsible for converting the scores into attention weights.

Let's first look at the dimensionality of our Keys:

In [44]:
d_k = keys.shape[-1]

print("Key dimensionality:", d_k)
print("Scaling factor:", d_k ** 0.5)

Key dimensionality: 3
Scaling factor: 1.7320508075688772


#### 3.4.1 Scaling One Attention Score

Let's look at one of our attention scores.

Suppose the attention score between two tokens is:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ score = 6 $

Since our Key vectors have 3 dimensions:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ d_k = 3 $

we divide the score by:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \sqrt{3} $

So:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \frac{6}{\sqrt{3}} \approx 3.46 $

The score has not been converted into a probability.

We have simply reduced its magnitude.

We can think of scaling as keeping the attention scores within a more manageable range before softmax.

In [45]:
score = 6
scaled_score = score / (3 ** 0.5)
print("Original score:", score)
print("Scaled score:", scaled_score)

Original score: 6
Scaled score: 3.464101615137755


#### 3.4.2 Scaling All Attention Scores

Now let's apply the same operation to the attention scores we calculated earlier.

Our attention score matrix contains the Query–Key dot products for every token:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ QK^T $

We divide every score by:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \sqrt{d_k} $

This gives us the scaled attention scores:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \frac{QK^T}{\sqrt{d_k}} $

In [46]:
# Get the dimensionality of the Key vectors.
d_k = keys.shape[-1]

# Scale every attention score by dividing by the square root
# of the Key dimensionality
scaled_attention_scores = attention_scores / (d_k ** 0.5)
print("Scaled attention scores:\n", scaled_attention_scores)

Scaled attention scores:
 tensor([[0.3986, 0.5100, 0.4209, 0.3619, 0.4293, 0.3415, 0.4084],
        [0.6164, 0.7905, 0.6534, 0.5595, 0.6664, 0.5288, 0.6334],
        [0.9296, 1.1760, 0.9544, 0.8427, 0.9802, 0.7811, 0.9332],
        [0.3988, 0.5087, 0.4182, 0.3619, 0.4272, 0.3401, 0.4065],
        [0.7108, 0.9058, 0.7421, 0.6448, 0.7594, 0.6036, 0.7223],
        [0.7630, 0.9606, 0.7752, 0.6914, 0.7977, 0.6372, 0.7601],
        [0.7543, 0.9565, 0.7788, 0.6840, 0.7988, 0.6362, 0.7603]],
       grad_fn=<DivBackward0>)


We can compare the original attention scores with the scaled scores.

The relative relationships between the scores are preserved, but their magnitudes are reduced.

This is exactly what we want.

We are not changing which tokens have higher or lower scores.

We are simply controlling the size of the values before passing them to softmax.

In [47]:
print("Original attention scores:\n", attention_scores)
print("\nScaled attention scores:\n", scaled_attention_scores)

Original attention scores:
 tensor([[0.6905, 0.8833, 0.7289, 0.6268, 0.7435, 0.5915, 0.7073],
        [1.0676, 1.3692, 1.1317, 0.9691, 1.1542, 0.9159, 1.0970],
        [1.6101, 2.0370, 1.6531, 1.4596, 1.6977, 1.3529, 1.6163],
        [0.6908, 0.8811, 0.7243, 0.6268, 0.7399, 0.5891, 0.7041],
        [1.2311, 1.5689, 1.2853, 1.1168, 1.3154, 1.0455, 1.2510],
        [1.3216, 1.6639, 1.3426, 1.1976, 1.3816, 1.1037, 1.3165],
        [1.3065, 1.6567, 1.3490, 1.1847, 1.3836, 1.1020, 1.3169]],
       grad_fn=<CopySlices>)

Scaled attention scores:
 tensor([[0.3986, 0.5100, 0.4209, 0.3619, 0.4293, 0.3415, 0.4084],
        [0.6164, 0.7905, 0.6534, 0.5595, 0.6664, 0.5288, 0.6334],
        [0.9296, 1.1760, 0.9544, 0.8427, 0.9802, 0.7811, 0.9332],
        [0.3988, 0.5087, 0.4182, 0.3619, 0.4272, 0.3401, 0.4065],
        [0.7108, 0.9058, 0.7421, 0.6448, 0.7594, 0.6036, 0.7223],
        [0.7630, 0.9606, 0.7752, 0.6914, 0.7977, 0.6372, 0.7601],
        [0.7543, 0.9565, 0.7788, 0.6840, 0.7988, 0.6362, 

#### 3.4.3 From Scaled Scores to Attention Weights

Now that the scores have been scaled, we can apply softmax.

The order is therefore:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ QK^T $

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; ↓

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \frac{QK^T}{\sqrt{d_k}} $

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; ↓

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \text{softmax} $

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; ↓

&nbsp;&nbsp; $ \text{Attention weights} $

This gives us the complete scaled attention calculation:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \text{Attention}(Q,K,V) = \text{softmax} \left(\frac{QK^T}{\sqrt{d_k}}\right)V $

Let's calculate the attention weights from our scaled scores.

In [48]:
# Convert the scaled attention scores into attention weights
# Softmax is applied across each row so that the weights
# for every query token sum to 1
attention_weights = torch.softmax(scaled_attention_scores, dim=-1)
print("Attention weights (after scaled attention scores):\n", attention_weights)

Attention weights (after scaled attention scores):
 tensor([[0.1411, 0.1577, 0.1442, 0.1360, 0.1454, 0.1332, 0.1424],
        [0.1397, 0.1663, 0.1450, 0.1320, 0.1469, 0.1280, 0.1421],
        [0.1401, 0.1792, 0.1436, 0.1284, 0.1473, 0.1208, 0.1406],
        [0.1413, 0.1577, 0.1440, 0.1361, 0.1453, 0.1332, 0.1424],
        [0.1400, 0.1701, 0.1444, 0.1311, 0.1470, 0.1258, 0.1416],
        [0.1413, 0.1722, 0.1431, 0.1316, 0.1463, 0.1246, 0.1409],
        [0.1404, 0.1719, 0.1439, 0.1309, 0.1468, 0.1248, 0.1413]],
       grad_fn=<SoftmaxBackward0>)


We can verify that each row still sums to 1:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \sum_j attention\_weight_{ij} = 1 $

Scaling happens **before** softmax, so the resulting values are still normalized into attention weights.

The complete flow is now:

Input representations

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; ↓

Queries, Keys, and Values

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; ↓

Query–Key dot products

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; ↓

**Scale by $ \sqrt{d_k} $**

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; ↓

&nbsp;&nbsp;&nbsp;&nbsp; Softmax

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; ↓

Attention weights

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; ↓

Weighted sum of Values

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; ↓

&nbsp;&nbsp;Context vectors

The scaling step is small mathematically, but it is an important part of the attention mechanism used in modern Transformer models.

### 3.5 Putting Scaled Dot-Product Attention Together

We’ll now combine the steps we have already implemented into one complete attention mechanism, while keeping each operation visible.

#### 3.5.1 From Separate Operations to One Attention Mechanism

So far, we have implemented the attention mechanism step by step:

1. Calculate Query–Key dot products to obtain attention scores.
2. Scale the scores by dividing by $ (\sqrt{d_k}) $.
3. Apply softmax to convert the scaled scores into attention weights.
4. Multiply each attention weight by its corresponding Value vector.
5. Sum the weighted Value vectors to produce context vectors.

We can express the entire operation mathematically as:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \text{Attention}(Q,K,V) = \text{softmax} \left(\frac{QK^T}{\sqrt{d_k}}\right)V $

Let's break down what this equation means:

* $ QK^T $: compares each Query with every Key.
* $ \sqrt{d_k} $: controls the magnitude of the attention scores.
* $ \operatorname{softmax} $: converts the scaled scores into attention weights.
* $ V $: supplies the information that gets combined using those weights.

The output is a set of contextualized representations. Each token now has a representation informed by the Value vectors it attended to.

We will reproduce this calculation using the operations we have already learned.


##### **Calculate the Query–Key scores**

In [49]:
# Calculate the dot product between every Query and every Key.
attention_scores = queries @ keys.T
print("Attention scores:\n", attention_scores)

Attention scores:
 tensor([[0.6905, 0.8833, 0.7289, 0.6268, 0.7435, 0.5915, 0.7073],
        [1.0676, 1.3692, 1.1317, 0.9691, 1.1542, 0.9159, 1.0970],
        [1.6101, 2.0370, 1.6531, 1.4596, 1.6977, 1.3529, 1.6163],
        [0.6908, 0.8811, 0.7243, 0.6268, 0.7399, 0.5891, 0.7041],
        [1.2311, 1.5689, 1.2853, 1.1168, 1.3154, 1.0455, 1.2510],
        [1.3216, 1.6639, 1.3426, 1.1976, 1.3816, 1.1037, 1.3165],
        [1.3065, 1.6567, 1.3490, 1.1847, 1.3836, 1.1020, 1.3169]],
       grad_fn=<MmBackward0>)


**Explanation:** Each row represents a Query token, and each column represents a Key token. The result is a $ 7×7 $ matrix for our seven-token sentence.

##### **Scale the scores**

In [50]:
# Get the dimensionality of the Key vectors.
d_k = keys.shape[-1]

# Scale the attention scores.
scaled_attention_scores = attention_scores / (d_k ** 0.5)
print("Scaled attention scores:\n", scaled_attention_scores)

Scaled attention scores:
 tensor([[0.3986, 0.5100, 0.4209, 0.3619, 0.4293, 0.3415, 0.4084],
        [0.6164, 0.7905, 0.6534, 0.5595, 0.6664, 0.5288, 0.6334],
        [0.9296, 1.1760, 0.9544, 0.8427, 0.9802, 0.7811, 0.9332],
        [0.3988, 0.5087, 0.4182, 0.3619, 0.4272, 0.3401, 0.4065],
        [0.7108, 0.9058, 0.7421, 0.6448, 0.7594, 0.6036, 0.7223],
        [0.7630, 0.9606, 0.7752, 0.6914, 0.7977, 0.6372, 0.7601],
        [0.7543, 0.9565, 0.7788, 0.6840, 0.7988, 0.6362, 0.7603]],
       grad_fn=<DivBackward0>)


**Explanation:** We divide each score by $ \sqrt{d_k} $ before applying softmax.

##### **Calculate the attention weights**

In [51]:
# Apply softmax across the Key dimension.
attention_weights = torch.softmax(scaled_attention_scores, dim=-1)
print("Attention weights:\n", attention_weights)

Attention weights:
 tensor([[0.1411, 0.1577, 0.1442, 0.1360, 0.1454, 0.1332, 0.1424],
        [0.1397, 0.1663, 0.1450, 0.1320, 0.1469, 0.1280, 0.1421],
        [0.1401, 0.1792, 0.1436, 0.1284, 0.1473, 0.1208, 0.1406],
        [0.1413, 0.1577, 0.1440, 0.1361, 0.1453, 0.1332, 0.1424],
        [0.1400, 0.1701, 0.1444, 0.1311, 0.1470, 0.1258, 0.1416],
        [0.1413, 0.1722, 0.1431, 0.1316, 0.1463, 0.1246, 0.1409],
        [0.1404, 0.1719, 0.1439, 0.1309, 0.1468, 0.1248, 0.1413]],
       grad_fn=<SoftmaxBackward0>)


**Explanation:** Each row now contains the attention weights for one Query token. The weights in each row sum to 1.

##### **Calculate the context vectors**

In [52]:
# Combine the Value vectors using the attention weights.
context_vectors = attention_weights @ values
print("Context vectors:\n", context_vectors)

Context vectors:
 tensor([[0.6584, 1.0262, 1.1260],
        [0.6595, 1.0284, 1.1289],
        [0.6597, 1.0304, 1.1302],
        [0.6583, 1.0261, 1.1257],
        [0.6595, 1.0290, 1.1291],
        [0.6588, 1.0287, 1.1278],
        [0.6593, 1.0291, 1.1289]], grad_fn=<MmBackward0>)


**Explanation:** Matrix multiplication performs the weighted sum of the Value vectors for every Query at once. Each row of `context_vectors` is the contextualized representation of a token.

##### **Check the output shape**

In [53]:
print("Input shape:", inputs.shape)
print("Query shape:", queries.shape)
print("Key shape:", keys.shape)
print("Value shape:", values.shape)
print("Attention weights shape:", attention_weights.shape)
print("Context vectors shape:", context_vectors.shape)

Input shape: torch.Size([7, 3])
Query shape: torch.Size([7, 3])
Key shape: torch.Size([7, 3])
Value shape: torch.Size([7, 3])
Attention weights shape: torch.Size([7, 7])
Context vectors shape: torch.Size([7, 3])


The context vectors should have shape `(7, 3)`: seven tokens, each represented by three values.

##### **What we have built**

We have now implemented the core mathematical operation of **scaled dot-product attention.**

One important distinction: our Query, Key, and Value projection matrices are currently manually initialized parameters. We have not trained them, so this is an implementation of the mechanism, not yet a trained attention layer.

The next natural step is to make this calculation reusable in a `SelfAttention` class, then introduce **causal masking** so a token cannot attend to future tokens when training an autoregressive language model.

### 3.6 Building a Reusable Self-Attention Class

So far, we have implemented scaled dot-product attention using separate code cells. Now we'll organize those operations into a reusable PyTorch module.

The goal is not to introduce new mathematics yet. It's to turn what we already understand into a component we can reuse.

#### 3.6.1 Why Do We Need a Self-Attention Class?

Our current implementation requires us to calculate the Query, Key, and Value projections, calculate the attention scores, scale them, apply softmax, and combine the Value vectors manually.

We want to package these operations into a single component.

In PyTorch, we can do this by creating a class that inherits from `torch.nn.Module`.

Our self-attention layer will:

1. Receive token representations as input.
2. Project them into Queries, Keys, and Values using trainable linear layers.
3. Calculate scaled Query–Key attention scores.
4. Apply softmax to obtain attention weights.
5. Combine the Value vectors to produce contextualized representations.

We'll begin with a single sequence, without causal masking or dropout. Those will be introduced separately so we can understand what each addition does.


##### **Define the class**

In [54]:
import torch
import torch.nn as nn

class SelfAttention(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()

        # Project the input representations into Queries, Keys, and Values.
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)

    def forward(self, x):
        # Project the input into Queries, Keys, and Values.
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        # Measure the similarity - Calculate the Query-Key dot products.
        attention_scores = queries @ keys.T

        # Scale - Scale the scores by the square root of the Key dimensionality.
        d_k = keys.shape[-1]
        scaled_attention_scores = attention_scores / (d_k ** 0.5)

        # Normalize - Convert the scaled scores into attention weights.
        attention_weights = torch.softmax(
            scaled_attention_scores,
            dim=-1
        )

        # Combine the Value vectors using the attention weights.
        context_vectors = attention_weights @ values

        return context_vectors

**Explanation**

Notice the difference from our earlier implementation:

- `nn.Linear` creates trainable projection layers. PyTorch initializes their weights.
- `forward()` defines the calculations performed when we pass data through the layer.
- We no longer need to manually create the projection matrices or write loops to calculate every attention score.
- The mathematical operations remain the same.

The `d_in` argument specifies the input representation size, while `d_out` specifies the output size of each Query, Key, and Value vector.

##### **Create the layer**

In [55]:
torch.manual_seed(123)

d_in = inputs.shape[-1]
print(f"Input representation size (d_in):\t\t  {d_in}\n")

d_out = 3
print(f"Output size of each Query, Key, and Value vector: {d_out}\n")

self_attention = SelfAttention(d_in, d_out)
print("Self Attention:\n", self_attention)

Input representation size (d_in):		  3

Output size of each Query, Key, and Value vector: 3

Self Attention:
 SelfAttention(
  (W_query): Linear(in_features=3, out_features=3, bias=False)
  (W_key): Linear(in_features=3, out_features=3, bias=False)
  (W_value): Linear(in_features=3, out_features=3, bias=False)
)


**Explanation:** Our illustrative input vectors have three dimensions, so `d_in = 3`. We also choose `d_out = 3` to keep the output dimensions easy to compare with the earlier examples.

The projection weights are newly initialized. Therefore, the resulting attention weights and context vectors will not necessarily match those from our earlier manually initialized projection matrices.

##### **Pass the tokens through the layer**

In [56]:
context_vectors = self_attention(inputs)
print("Input shape: ", inputs.shape)
print("Output shape:", context_vectors.shape)
print("Context vectors:\n", context_vectors)

Input shape:  torch.Size([7, 3])
Output shape: torch.Size([7, 3])
Context vectors:
 tensor([[ 0.2241,  0.4155, -0.1692],
        [ 0.2234,  0.4144, -0.1675],
        [ 0.2221,  0.4134, -0.1669],
        [ 0.2240,  0.4155, -0.1695],
        [ 0.2229,  0.4142, -0.1678],
        [ 0.2226,  0.4142, -0.1680],
        [ 0.2227,  0.4141, -0.1677]], grad_fn=<MmBackward0>)


The output should have shape `(7, 3)`.

Each of the seven tokens now has a contextualized representation produced by our reusable attention layer.

##### **Inspect the learned projection weights**

In [57]:
print("Query projection weights:\n", self_attention.W_query.weight)
print("\nKey projection weights:\n", self_attention.W_key.weight)
print("\nValue projection weights:\n", self_attention.W_value.weight)

Query projection weights:
 Parameter containing:
tensor([[-0.2354,  0.0191, -0.2867],
        [ 0.2177, -0.4919,  0.4232],
        [-0.4196, -0.4590, -0.3648]], requires_grad=True)

Key projection weights:
 Parameter containing:
tensor([[ 0.2615, -0.2133,  0.2161],
        [-0.4900, -0.3503, -0.2120],
        [-0.1135, -0.4404,  0.3780]], requires_grad=True)

Value projection weights:
 Parameter containing:
tensor([[-0.1362,  0.1853,  0.4083],
        [ 0.1076,  0.1579,  0.5573],
        [-0.2604,  0.1829, -0.2569]], requires_grad=True)


These matrices are **trainable parameters**. They are initialized here, but they have not yet been trained on a language-model objective.

During training, backpropagation and an optimizer update these weights so that the attention mechanism becomes useful for the task.

##### **What comes next?**

Our current class allows every token to attend to every token in the sequence—including future tokens.

That is unsuitable for the usual autoregressive language-model training objective, where the model predicts the next token without seeing future tokens.

**Next: causal masking.** We'll first understand the problem using our sentence, *“The bank said there was no network,”* then implement a mask that prevents each token from attending to later tokens.

### 3.7 Causal Masking

We have a working self-attention layer. However, there is a problem: **every token can currently attend to every other token, including tokens that come after it.**

Let's understand why this matters before changing our code.

#### 3.7.1 Why Do We Need Causal Masking?

Consider our sentence:

**“The bank said there was no network.”**

Suppose a language model is learning to predict the next token.

When it processes the token *“The”*, it should only have access to information available up to that position. It must not use *“bank”*, *“said”*, or any later token to help make its prediction.

Similarly, when processing *“said”*, it can use *“The”*, *“bank”*, and *“said”*, but it must not look ahead at *“there was no network”*.

Our current self-attention implementation does not enforce this restriction. Every Query can attend to all Keys in the sequence.

**Causal masking** solves this problem by preventing each token from attending to future tokens.

This is essential for autoregressive language models, which learn to predict the next token using only the preceding context.


#### 3.7.2 Understand the mask mathematically

We can represent the allowed and blocked positions using a lower-triangular matrix.

```text
[
 [1, 0, 0, 0, 0, 0, 0],
 [1, 1, 0, 0, 0, 0, 0],
 [1, 1, 1, 0, 0, 0, 0],
 [1, 1, 1, 1, 0, 0, 0],
 [1, 1, 1, 1, 1, 0, 0],
 [1, 1, 1, 1, 1, 1, 0],
 [1, 1, 1, 1, 1, 1, 1]
]
```

- `1` means the attention connection is allowed.
- `0` means the connection is blocked.

We do not apply this matrix directly to the attention weights. Instead, we use it to modify the **attention scores before softmax.**

For blocked positions, we replace the scores with negative infinity:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $
s_{ij} = \begin{cases}
s_{ij}, & j \le i \\
-\infty, & j > i
\end{cases}
$

Why negative infinity? Because:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ e^{-\infty} $

After softmax, blocked positions receive zero attention weight.

#### 3.7.3 Create the causal mask

In [58]:
# Get the number of tokens in the input sequence.
sequence_length = inputs.shape[0]

# Create a lower-triangular matrix of ones.
# Position on and below the diagonal are allowed.
# Positions above the diagonal represent future tokens.
causal_mask = torch.tril(torch.ones(sequence_length, sequence_length))
print("Causal mask:\n", causal_mask)

Causal mask:
 tensor([[1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1.]])


**Explanation:** `torch.tril()` keeps the lower triangle of a matrix and sets the upper triangle to zero.

#### 3.7.4 Inspect the mask

In [59]:
print("Tokens:\n", tokens)
print("\nCausal mask:\n", causal_mask)

Tokens:
 ['The', 'bank', 'said', 'there', 'was', 'no', 'network']

Causal mask:
 tensor([[1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1.]])


Compare the resulting matrix with the allowed-attention pattern above.

#### 3.7.5 Apply the mask to attention scores

We already know how to calculate scaled attention scores:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ S = \frac{QK^T}{\sqrt{d_k}} $

Now we want to prevent future positions from contributing to attention.

We will use `masked_fill()` to replace the blocked scores with negative infinity. This method changes the values at positions selected by a Boolean mask.

#### 3.7.6 Mask future positions

In [60]:
# Calculate the Query-Key scores.
attention_scores = queries @ keys.T

# Scale the scores
d_k = keys.shape[-1]
scaled_attention_scores = attention_scores / (d_k ** 0.5)

# Identify positions that correspond to future tokens.
# Convert the zeros in the causal mask to True.
blocked_positions = causal_mask == 0
print("Block positions:\n", blocked_positions)

# Replace future-token scores with negative infinity.
masked_attention_scores = scaled_attention_scores.masked_fill(
    blocked_positions,
    float("-inf")
)
print("\nMasked attention scores:\n", masked_attention_scores)

Block positions:
 tensor([[False,  True,  True,  True,  True,  True,  True],
        [False, False,  True,  True,  True,  True,  True],
        [False, False, False,  True,  True,  True,  True],
        [False, False, False, False,  True,  True,  True],
        [False, False, False, False, False,  True,  True],
        [False, False, False, False, False, False,  True],
        [False, False, False, False, False, False, False]])

Masked attention scores:
 tensor([[0.3986,   -inf,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.6164, 0.7905,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.9296, 1.1760, 0.9544,   -inf,   -inf,   -inf,   -inf],
        [0.3988, 0.5087, 0.4182, 0.3619,   -inf,   -inf,   -inf],
        [0.7108, 0.9058, 0.7421, 0.6448, 0.7594,   -inf,   -inf],
        [0.7630, 0.9606, 0.7752, 0.6914, 0.7977, 0.6372,   -inf],
        [0.7543, 0.9565, 0.7788, 0.6840, 0.7988, 0.6362, 0.7603]],
       grad_fn=<MaskedFillBackward0>)


**Explanation:** The entries above the diagonal should now display `-inf`. These scores will not contribute to the corresponding token's attention distribution after softmax.

#### 3.7.7 Apply softmax

In [61]:
# Convert the masked scores into attention weights.
attention_weights = torch.softmax(masked_attention_scores, dim=-1)
print("Attention weights:\n", attention_weights)

Attention weights:
 tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4566, 0.5434, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3026, 0.3872, 0.3102, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2439, 0.2723, 0.2487, 0.2351, 0.0000, 0.0000, 0.0000],
        [0.1911, 0.2322, 0.1972, 0.1789, 0.2006, 0.0000, 0.0000],
        [0.1645, 0.2005, 0.1665, 0.1531, 0.1703, 0.1451, 0.0000],
        [0.1404, 0.1719, 0.1439, 0.1309, 0.1468, 0.1248, 0.1413]],
       grad_fn=<SoftmaxBackward0>)


Check the result carefully: the weights assigned to future tokens should be zero.

#### 3.7.8 Verify that future tokens receive zero weight

In [62]:
# Check the weights at positions that were blocked by causal mask.
print(attention_weights[blocked_positions])

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       grad_fn=<IndexBackward0>)


Every returned value should be zero.

We can also verify that every row sums to one:

In [63]:
print("Row sums:\n", attention_weights.sum(dim=-1))

Row sums:
 tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       grad_fn=<SumBackward1>)


Each row should sum to approximately `1.0`, allowing for normal floating-point precision.

#### 3.7.9 Calculate the context vectors

The final operation is unchanged. We combine the Value vectors using the masked attention weights.

In [64]:
# Combine the Value vectors using the causal attention weights.
context_vectors = attention_weights @ values
print("Context vectors:\n", context_vectors)

Context vectors:
 tensor([[0.4759, 0.8924, 0.7463],
        [0.5831, 1.0269, 0.9870],
        [0.6697, 1.0754, 1.1537],
        [0.6197, 1.0155, 1.0500],
        [0.6382, 1.0260, 1.0992],
        [0.6477, 1.0225, 1.1058],
        [0.6593, 1.0291, 1.1289]], grad_fn=<MmBackward0>)


#### 3.7.10 What have we achieved?

Our attention mechanism now respects the left-to-right order required by autoregressive language models.

The complete flow is:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $
QK^T \rightarrow 
\frac{QK^T}{\sqrt{d_k}} \rightarrow 
\text{Causal Masking} \rightarrow 
\text{Softmax} \rightarrow
\text{Weighted Values}
$

One important clarification: causal masking does not change the Query, Key, or Value projections. It changes which attention connections are allowed.

**Next:** we'll introduce **dropout**, a regularization technique that helps reduce overfitting during training.

### 3.8 Adding Dropout to Self-Attention

We have now implemented causal masking, which prevents tokens from attending to future tokens. The next step is to introduce **dropout**, a regularization technique used during training.

#### 3.8.1 What Is Dropout?

During training, a neural network can become too dependent on particular patterns in its training data. This can lead to overfitting, where the model performs well on training examples but struggles to generalize to new examples.

Dropout helps reduce this problem by randomly setting some values to zero during training.

In our attention mechanism, we apply dropout to the attention weights.

Recall that attention weights determine how much each token attends to other tokens. During training, dropout randomly sets some of these weights to zero, reducing the model's reliance on any single attention connection.

For example, suppose the attention weights for a token are:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $
[0.1,\ 0.2,\ 0.6,\ 0.1]
$

With dropout, some entries may temporarily become zero during a training pass.

Dropout does not decide which tokens are in the future. **Causal masking prevents future-token access; dropout is used to regularize training.**

PyTorch's dropout layer also scales the surviving values during training so that their expected magnitude remains consistent. Therefore, after dropout, the weights in a row are not necessarily guaranteed to sum to one.

#### 3.8.2 Understand dropout with a simple example

Let's start with a small vector of attention weights before applying dropout.

In [65]:
# Create a small example of attention weights.
example_weights = torch.tensor([0.1, 0.2, 0.6, 0.1])

print("Original weights:", example_weights)
print("Sum:\t\t ", example_weights.sum())

Original weights: tensor([0.1000, 0.2000, 0.6000, 0.1000])
Sum:		  tensor(1.)


The weights sum to one because they were produced by softmax.

#### 3.8.3 Apply dropout

In [66]:
# Create a dropout layer with a probability of 0.5.
dropout = torch.nn.Dropout(p=0.5)

# Apply dropout to the example weights.
dropped_weights = dropout(example_weights)

print("Original weights:\n", example_weights)
print("\nWeights after dropout:\n", dropped_weights)

Original weights:
 tensor([0.1000, 0.2000, 0.6000, 0.1000])

Weights after dropout:
 tensor([0.0000, 0.4000, 0.0000, 0.0000])


You may see some weights become zero. The nonzero weights are scaled by $ 1/(1−p) $. With $ p=0.5 $, surviving values are multiplied by 2.

Your exact output can differ between runs because dropout randomly selects which values to remove during training.

#### 3.8.4 Why the weights may no longer sum to one

Before dropout, softmax ensures:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \sum_ja_{ij} = 1 $

After dropout, some weights are zeroed and the surviving weights are rescaled. Their sum is therefore not necessarily one.

This is expected. Dropout is a training-time regularization operation, not another softmax normalization step.

In PyTorch, dropout behaves differently depending on the mode:
- **Training mode:** randomly drops values and scales those that survive.
- **Evaluation mode:** does not drop values; it acts as the identity operation.

#### 3.8.5 Compare training and evaluation modes

In [67]:
dropout = torch.nn.Dropout(p=0.5)

# Training mode: dropout is active.
dropout.train()
print("Training mode:  ", dropout(example_weights))

# Evaluation mode: dropout is disabled.
dropout.eval()
print("Evaluation mode:", dropout(example_weights))

Training mode:   tensor([0.2000, 0.0000, 1.2000, 0.0000])
Evaluation mode: tensor([0.1000, 0.2000, 0.6000, 0.1000])


In evaluation mode, the output matches the original input.

### 3.9 Add dropout to our self-attention layer

We will now update the structure of our attention layer conceptually. We won't introduce batching or multi-head attention yet.

#### 3.9.1 Where does dropout belong?

Our current attention flow is:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; 
$
QK^T \rightarrow 
\frac{QK^T}{\sqrt{d_k}} \rightarrow 
\text{Causal Masking} \rightarrow 
\text{Softmax} \rightarrow
\text{Weighted Values}
$

With dropout, the flow becomes:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; 
$
QK^T \rightarrow 
\frac{QK^T}{\sqrt{d_k}} \rightarrow 
\text{Causal Masking} \rightarrow 
\text{Softmax} \rightarrow
\boxed{\text{Dropout}} \rightarrow
\text{Weighted Values}
$

Dropout is applied **after softmax and before multiplying the attention weights by the Value vectors.**

This preserves causal masking: the future-token positions already have zero attention weight, and dropout does not make those blocked positions attend to future tokens.

**A note before we update the class**

We now understand the role of dropout and how PyTorch implements it. When we incorporate it into our reusable SelfAttention class, we will need to preserve the causal mask as well.

That brings together three distinct ideas:
- **Scaling** controls the magnitude of attention scores.
- **Causal masking** prevents attention to future tokens.
- **Dropout** regularizes attention during training.

Next, we can integrate dropout into the causal self-attention class, then move on to handling batches of input sequences.

### 3.10 Integrating Causal Masking and Dropout

So far, we have learned how to calculate scaled attention scores, prevent attention to future tokens, and apply dropout during training. Now let's bring these pieces together in our reusable SelfAttention class.

The goal is to preserve the mathematical steps we've learned while making the implementation reusable.

#### 3.10.1 What are we adding?

Our attention mechanism now follows this sequence:
1. Project the input representations into Queries, Keys, and Values.
2. Calculate Query-Key dot products.
3. Scale the scores by $ \sqrt{d_k} $.
4. Apply causal masking to block future tokens.
5. Apply softmax to obtain attention weights.
6. Apply dropout to the attention weights.
7. Multiply the resulting weights by Value vectors.

The key distinction is that **causal masking controls which tokens can be attended to, while dropout randomly removes some attention weights during training**.

#### 3.10.2 Create the causal self-attention class

In [68]:
import torch
import torch.nn as nn

class CausalSelfAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout):
        super().__init__()

        # Project input representations into Queries, Keys, and Values.
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)

        # Dropout applied to the attention weights
        self.dropout = nn.Dropout(dropout)

        # Creat a mask that blocks future-token positions.
        mask = torch.triu(
            torch.ones(context_length, context_length),
            diagonal=1
        )

        # Store the mask as part of the module.
        # PyTorch will move it with the module but won't train it
        self.register_buffer("mask", mask)

    def forward(self, x):
        # Get the numer of tokens in this input sequence.
        sequence_length = x.shape[0]

        # Calculate Queries, Keys, and Values.
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        # Calculate and scale the Query-Key scores.
        attention_scores = queries @ keys.T
        d_k = keys.shape[-1]
        attention_scores = attention_scores / (d_k ** 0.5)

        # Block future-token positions with negative infinity.
        attention_scores = attention_scores.masked_fill(
            self.mask[:sequence_length, :sequence_length].bool(),
            -torch.inf
        )

        # Convert scores into attention weights.
        attention_weights = torch.softmax(attention_scores, dim=-1)

        # Apply dropout to the attention weights.
        attention_weights = self.dropout(attention_weights)

        # Combine the Value vectors using the attention weights.
        context_vectors = attention_weights @ values

        return context_vectors        

**Explanation**

A few new details are worth understanding:
- `nn.Dropout(dropout)` creates a dropout layer with specified probability.
- `torch.triu(..., diagonal=1)` creates a mask identifying future positions.
- `register_buffer()` stores the mask as part of the module without making it a trainable parameters.
  - If not used, it would stay on the CPU, causing the code to crash with a device mismatch error if the model is moved to a GPU using `model.cuda()`.
  - If the model is moved to the GPU, the mask automatically moves to the GPU with it. This avoids device mismatch error.
- `self.mask[:sequence_length, :sequence_length]` selects the relevant part of the mask for the current sequence length.

This version handles **one sequence at a time**, matching our current notebook progression. We will extend it to batches next.

#### 3.10.3 Instantiate the layer

In [69]:
torch.manual_seed(123)

d_in = inputs.shape[-1]
d_out = 3
context_length = len(tokens)
dropout_rate = 0.2

causal_attention = CausalSelfAttention(
    d_in=d_in,
    d_out=d_out,
    context_length=context_length,
    dropout=dropout_rate
)

print("Causal attention:\n", causal_attention)

Causal attention:
 CausalSelfAttention(
  (W_query): Linear(in_features=3, out_features=3, bias=False)
  (W_key): Linear(in_features=3, out_features=3, bias=False)
  (W_value): Linear(in_features=3, out_features=3, bias=False)
  (dropout): Dropout(p=0.2, inplace=False)
)


#### 3.10.4 Run it in training mode

In [70]:
# Enable training mode so dropout is active.
causal_attention.train()

print("Inputs:\n", inputs)

context_vectors = causal_attention(inputs)

print("\nInput shape: ", inputs.shape)
print("Output shape:", context_vectors.shape)

print("\nContext vectors\n:", context_vectors)

Inputs:
 tensor([[0.4200, 0.1800, 0.7600],
        [0.8100, 0.3400, 0.5900],
        [0.6300, 0.9100, 0.2700],
        [0.3700, 0.2200, 0.6800],
        [0.7400, 0.5600, 0.3100],
        [0.2900, 0.7900, 0.4700],
        [0.5800, 0.6700, 0.3900]])

Input shape:  torch.Size([7, 3])
Output shape: torch.Size([7, 3])

Context vectors
: tensor([[ 0.0000,  0.0000,  0.0000],
        [ 0.1232,  0.2987, -0.1910],
        [ 0.2754,  0.5446, -0.2516],
        [ 0.2069,  0.4051, -0.1842],
        [ 0.2631,  0.5245, -0.2514],
        [ 0.2082,  0.3639, -0.1657],
        [ 0.2333,  0.4394, -0.1668]], grad_fn=<MmBackward0>)


The output should have shape `(7, 3)`: one three-dimensional contextual representation for each token.

Because dropout is active, the results can vary between training passes.

#### 3.10.5 Run it in evaluation mode

In [71]:
# Disable dropout for evaluation.
causal_attention.eval()

context_vectors = causal_attention(inputs)
print("Context vectors:\n", context_vectors)

Context vectors:
 tensor([[ 0.2865,  0.4971, -0.2717],
        [ 0.2392,  0.4831, -0.2862],
        [ 0.2203,  0.4357, -0.2013],
        [ 0.2337,  0.4425, -0.2120],
        [ 0.2105,  0.4196, -0.2011],
        [ 0.2258,  0.4192, -0.1750],
        [ 0.2227,  0.4141, -0.1677]], grad_fn=<MmBackward0>)


In evaluation mode, dropout is disabled. Causal masking remains active, so future-token attention is still blocked.

#### 3.10.6 What have we achieved?

Our class now combines the core operations of causal self-attention with dropout.

One limitation remains: the implementation expects input shaped as (sequence_length, embedding_dimension). Real language-model training processes multiple sequences together in batches.

**Next, we'll introduce batch dimensions** and learn how the attention calculations change when we process several sequences at once.

### 3.11 Extending Self-Attention to Batches

So far, our `CausalSelfAttention` class processes one sequence at a time. In practice, language models typically process multiple sequences together in a **batch**.

Let's understand what changes and why.

#### 3.11.1 Processing Multiple Sequences at Once

Imagine training a language model using these two sequences:

* “The bank said there was no network.”
* “The server stopped working today.”

Instead of processing one sequence, updating the model, and then processing the next, we can group multiple sequences into a batch.

Batching lets us process several sequences in one forward pass, making better use of the hardware available for training and inference.

To represent a batch, we add another dimension to our input tensor.

Previously, our input had the shape:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ (T, D) $

where:

* \(T\) is the sequence length.
* \(D\) is the input embedding dimension.

With batches, the shape becomes:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ (B, T, D) $

where \(B\) is the batch size.

Each batch contains multiple sequences, and each sequence contains multiple token representations.

Our next goal is to adapt self-attention so it can process this three-dimensional input.


#### 3.11.2 Create a batch of illustrative inputs

We'll use our existing `inputs` tensor as the first sequence. For this exercise, we'll create a second sequence by reusing the same illustrative representations. This keeps the focus on tensor shapes rather than introducing another set of values.

In [72]:
# Our original input contains one sequence.
print("Original input shape:", inputs.shape)

# Add a batch and duplicate the sequence.
batch_inputs = torch.stack([inputs, inputs], dim=0)
print("Batched input shape: ", batch_inputs.shape)

Original input shape: torch.Size([7, 3])
Batched input shape:  torch.Size([2, 7, 3])


**Explanation**

The shape `(2, 7, 3)` means:

- 2 sequences in the batch.
- 7 tokens in each sequence.
- 3 values in each token representation.

We duplicated the sequence only to demonstrate batching. Real training batches would typically contain different sequences.

#### 3.11.3 What changes in the attention calculation?

For a single sequence, we calculated attention scores with:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;  $ S = QK^T $

For a batch, we want to perform the same calculation independently for every sequence.

The key difference is that we must transpose the **last two dimensions** of the Key tensor, not the entire tensor.

For batched tensors:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ Q \in \mathbb{R}^{B \times T \times D_k} $

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ K \in \mathbb{R}^{B \times T \times D_k} $

we calculate:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ S = QK^T $

with a batched matrix multiplication, producing:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ S \in \mathbb{R}^{B \times T \times T} $

Each sequence gets its own attention-score matrix. No sequence attends to tokens in another sequence.

#### 3.11.4 Update the class for batches

The main changes are:
- Use `x.shape[1]` to obtain the sequence length.
- Use `keys.transpose(1, 2)` to transpose the Key tensor's sequence and feature dimensions.
- Keep the causal mask aligned with the sequence dimensions.

In [73]:
import torch
import torch.nn as nn

class CausalSelfAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout):
        super().__init__()

        # Project input representations into Queries, Keys, and Values.
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)

        # Dropout applied to the attention weights
        self.dropout = nn.Dropout(dropout)

        # Create a mask that blocks future-token positions.
        mask = torch.triu(
            torch.ones(context_length, context_length),
            diagonal=1
        )

        self.register_buffer("mask", mask)

    def forward(self, x):
        # x has shape (batch_size, sequence_length, input_dimension).
        batch_size, sequence_length, _ = x.shape

        # Project the input into Queries, keys, and Values.
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        # Calculate attention scores independently for each sequence.
        # keys.transpose(1, 2) has shape (batch_size, d_out, sequence_length).
        attention_scores = queries @ keys.transpose(1, 2)

        # Scale the scores by the square root of the Key dimensionality.
        d_k = keys.shape[-1]
        attention_scores = attention_scores / (d_k ** 0.5)

        # Block future-token positions.
        attention_scores = attention_scores.masked_fill(
            self.mask[:sequence_length, :sequence_length].bool(),
            -torch.inf
        )

        # Convert scores into attention weights.
        attention_weights = torch.softmax(attention_scores, dim=-1)

        # Apply dropout to the attention weights.
        attention_weights = self.dropout(attention_weights)

        # Combine the Value vectors using the attention weights.
        context_vectors = attention_weights @ values

        return context_vectors

Explanation

Notice that the linear layers work with batched input automatically. They transform the final dimension while preserving the batch and sequence dimensions.

The important line is:

```python
attention_scores = queries @ keys.transpose(1, 2)
```

Each batch item has its own Query and Key matrices, and PyTorch performs the matrix multiplication for each batch item.

#### 3.11.5 Run batched self-attention

In [74]:
torch.manual_seed(123)

causal_attention = CausalSelfAttention(
    d_in=3,
    d_out=3,
    context_length=7,
    dropout=0.2
)

# Disable dropout so the demonstration is deterministic
causal_attention.eval()

batch_context_vectors = causal_attention(batch_inputs)
print("Batched input shape:", batch_inputs.shape)
print("Output shape:\t    ", batch_context_vectors.shape)

Batched input shape: torch.Size([2, 7, 3])
Output shape:	     torch.Size([2, 7, 3])


#### 3.11.6 Inspect the output for each sequence

In [75]:
print("Context vectors for sequence 1:")
print(batch_context_vectors[0])

print("\nContext vectors for sequence 2:")
print(batch_context_vectors[1])

Context vectors for sequence 1:
tensor([[ 0.2865,  0.4971, -0.2717],
        [ 0.2392,  0.4831, -0.2862],
        [ 0.2203,  0.4357, -0.2013],
        [ 0.2337,  0.4425, -0.2120],
        [ 0.2105,  0.4196, -0.2011],
        [ 0.2258,  0.4192, -0.1750],
        [ 0.2227,  0.4141, -0.1677]], grad_fn=<SelectBackward0>)

Context vectors for sequence 2:
tensor([[ 0.2865,  0.4971, -0.2717],
        [ 0.2392,  0.4831, -0.2862],
        [ 0.2203,  0.4357, -0.2013],
        [ 0.2337,  0.4425, -0.2120],
        [ 0.2105,  0.4196, -0.2011],
        [ 0.2258,  0.4192, -0.1750],
        [ 0.2227,  0.4141, -0.1677]], grad_fn=<SelectBackward0>)


Because we deliberately duplicated the inputs and used the same model parameters in evaluation mode, the two outputs should be identical.

#### 3.11.7 What have we achieved?

Our self-attention layer now supports batched sequences with input shape:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ (B,T,D) $

and produces contextualized representations of shape:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ (B,T,D_{out}) $

Causal masking and dropout still work as before, but attention is calculated independently for each sequence in the batch.

**Next, we'll move to multi-head attention**, where the model learns several separate Query–Key–Value projection sets and combines their outputs. This will build directly on the single-head mechanism we've now implemented.

### 3.12 Introducing Multi-Head Attention

So far, we have built a causal self-attention mechanism that calculates one set of attention scores for each token, converts them into attention weights, and combines Value vectors to produce contextual representations.

Now we'll explore how Transformers extend this mechanism using **multiple attention heads**.

#### 3.12.1 From Single-Head to Multi-Head Attention

Our current self-attention mechanism uses one set of Query, Key, and Value projections.

For each token, it calculates how strongly that token should attend to other tokens and combines their Value vectors accordingly.

But language contains many different kinds of relationships. A token may relate to another token because of grammatical structure, meaning, context, or other patterns.

One attention mechanism may learn useful relationships, but we can give the model multiple attention heads, each with its own Query, Key, and Value projections.

Each head independently calculates attention and produces its own contextual representations.

The outputs of the heads are then combined to create the final representation.

The standard multi-head attention operation is:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $
\operatorname{MultiHead}(Q,K,V)
=
\operatorname{Concat}(\operatorname{head}_1,\ldots,\operatorname{head}_h)W_O
$

where each head is calculated as:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $
\operatorname{head}_i
=
\operatorname{Attention}(Q_i,K_i,V_i)
$

The heads can learn different relationships, although we should not assume that every head will learn a distinct or easily interpretable role.


#### 3.12.2 Understand the architecture

<img src="./illustrations/3_11 Understand the architecture.png" width="700px">

Each head has its own learned projections. The heads process the same input, but their projections allow them to calculate attention differently.

#### 3.12.3 Start with dimensions

In [76]:
batch_inputs.shape

torch.Size([2, 7, 3])

This means two sequences, seven tokens per sequence, and three input features per token.

For our first multi-head example, we'll use:
- Input dimension: $ d_{in} = 3 $
- Number of heads: $ h = 3 $
- Output dimension per head: $ d_{head} = 2 $

Each head will produce two features per token. Concatenating three heads will therefore produce six features per token:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ d_{concat} = h \times d_{head} $

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ d_{concat} = 3 \times 2 = 6 $

We'll use these small dimensions to make the mechanics easy to inspect.

#### 3.12.4 Create three independent attention heads

In [77]:
# Set the dimensions for our multi-head attention example.
d_in = batch_inputs.shape[-1]
num_heads = 3
d_head = 2

# Create one independent causal self-attention module per head.
heads = nn.ModuleList([
    CausalSelfAttention(
        d_in=d_in,
        d_out=d_head,
        context_length=7,
        dropout=0.2
    )
    for _ in range(num_heads)
])

print("Input dimension:", d_in)
print("Number of heads:", num_heads)
print("Output dimension per head:", d_head)

Input dimension: 3
Number of heads: 3
Output dimension per head: 2


**Explanation**

`nn.ModuleList` registers each head as part of the model, allowing PyTorch to track its trainable parameters.

Each `CausalSelfAttention` instance has its own Query, Key, and Value projection weights. That means the heads can learn different projections during training.

We are using separate modules here to make the concept visible. Optimized implementations often combine the projections into larger matrix operations.

#### 3.12.5 Run each head

In [78]:
# Disable dropout so the demonstration is deterministic.
for head in heads:
    head.eval()

# Calculate the contextual representations produced by each head.
head_outputs = [head(batch_inputs) for head in heads]

for i, output in enumerate(head_outputs):
    print(f"Head {i + 1} output:\n", output)
    print(f"Head {i + 1} output shape:\n", output.shape)
    print()

Head 1 output:
 tensor([[[-0.2387,  0.0418],
         [-0.2141,  0.0784],
         [-0.1930,  0.2167],
         [-0.2019,  0.1677],
         [-0.1853,  0.1934],
         [-0.1918,  0.2263],
         [-0.1886,  0.2413]],

        [[-0.2387,  0.0418],
         [-0.2141,  0.0784],
         [-0.1930,  0.2167],
         [-0.2019,  0.1677],
         [-0.1853,  0.1934],
         [-0.1918,  0.2263],
         [-0.1886,  0.2413]]], grad_fn=<UnsafeViewBackward0>)
Head 1 output shape:
 torch.Size([2, 7, 2])

Head 2 output:
 tensor([[[-0.1423, -0.2905],
         [-0.0053, -0.1895],
         [ 0.0632, -0.0236],
         [ 0.0163, -0.0729],
         [ 0.0591, -0.0284],
         [ 0.0425,  0.0008],
         [ 0.0539,  0.0220]],

        [[-0.1423, -0.2905],
         [-0.0053, -0.1895],
         [ 0.0632, -0.0236],
         [ 0.0163, -0.0729],
         [ 0.0591, -0.0284],
         [ 0.0425,  0.0008],
         [ 0.0539,  0.0220]]], grad_fn=<UnsafeViewBackward0>)
Head 2 output shape:
 torch.Size([2, 7, 2

Each head processes both sequences and produces two features for every token.

#### 3.12.6 Concatenate the head outputs

We now have three separate output tensors. The next step is to concatenate their feature dimensions.

For each token, we combine the two features from Head 1, the two from Head 2, and the two from Head 3.

The resulting representation has six features.

In [79]:
# Concatenate the head outputs along the final feature dimension.
concatenated = torch.cat(head_outputs, dim=-1)
print("Concatenated head outputs:\n", concatenated)
print("\nConcatenated output shape:", concatenated.shape)

Concatenated head outputs:
 tensor([[[-0.2387,  0.0418, -0.1423, -0.2905, -0.3807,  0.2624],
         [-0.2141,  0.0784, -0.0053, -0.1895, -0.3094,  0.2656],
         [-0.1930,  0.2167,  0.0632, -0.0236, -0.1813,  0.3182],
         [-0.2019,  0.1677,  0.0163, -0.0729, -0.2099,  0.3063],
         [-0.1853,  0.1934,  0.0591, -0.0284, -0.1788,  0.3001],
         [-0.1918,  0.2263,  0.0425,  0.0008, -0.1593,  0.3236],
         [-0.1886,  0.2413,  0.0539,  0.0220, -0.1451,  0.3281]],

        [[-0.2387,  0.0418, -0.1423, -0.2905, -0.3807,  0.2624],
         [-0.2141,  0.0784, -0.0053, -0.1895, -0.3094,  0.2656],
         [-0.1930,  0.2167,  0.0632, -0.0236, -0.1813,  0.3182],
         [-0.2019,  0.1677,  0.0163, -0.0729, -0.2099,  0.3063],
         [-0.1853,  0.1934,  0.0591, -0.0284, -0.1788,  0.3001],
         [-0.1918,  0.2263,  0.0425,  0.0008, -0.1593,  0.3236],
         [-0.1886,  0.2413,  0.0539,  0.0220, -0.1451,  0.3281]]],
       grad_fn=<CatBackward0>)

Concatenated output shape:

The `dim=-1` argument means we concatenate along the last dimension, which contains the features.

We do not concatenate along the batch or token dimensions because we want to combine the heads' representations of the *same* tokens.

#### 3.12.7 Why do we need an output projection?

Concatenation combines the outputs, but it does not learn how the information from different heads should interact.

Transformers therefore typically apply a learned output projection:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ Y = \text{Concat}(\text{head}_1,...,\text{head}_h)W^T_O $

This projection can combine information across the concatenated features and map the result to the desired output dimension.

For this example, we'll project the six concatenated features back to three features per token.

In [80]:
# Create an output projection that maps six features back to three.
W_output = nn.Linear(num_heads * d_head, d_in, bias=False)

# Apply the output projection to the concatenated head outputs.
multi_head_output = W_output(concatenated)
print("Multi-head output:\n", multi_head_output)

print("\nConcatenated shape:", concatenated.shape)
print("Final output shape:", multi_head_output.shape)

Multi-head output:
 tensor([[[ 0.0640, -0.0873, -0.2209],
         [ 0.0275, -0.0599, -0.1471],
         [-0.0338, -0.0167, -0.0388],
         [-0.0176, -0.0325, -0.0690],
         [-0.0302, -0.0200, -0.0405],
         [-0.0424, -0.0172, -0.0243],
         [-0.0496, -0.0126, -0.0110]],

        [[ 0.0640, -0.0873, -0.2209],
         [ 0.0275, -0.0599, -0.1471],
         [-0.0338, -0.0167, -0.0388],
         [-0.0176, -0.0325, -0.0690],
         [-0.0302, -0.0200, -0.0405],
         [-0.0424, -0.0172, -0.0243],
         [-0.0496, -0.0126, -0.0110]]], grad_fn=<UnsafeViewBackward0>)

Concatenated shape: torch.Size([2, 7, 6])
Final output shape: torch.Size([2, 7, 3])


Our multi-head attention output now has the same feature dimension as our original input.

#### 3.12.8 What have we built?

We've constructed a simple multi-head attention mechanism from separate causal attention modules:

1. Each head independently projects the inputs into Queries, Keys, and Values.
2. Each head calculates scaled, causally masked attention.
3. Each head produces contextual representations.
4. We concatenate the outputs along the feature dimension.
5. We apply an output projection to combine information across the heads.

This example uses three heads with two features each. It is intentionally small, and it is not yet a fully optimized implementation.

**Next:** we'll examine how to combine the heads more efficiently by splitting a single set of projected features into multiple heads, rather than creating a separate attention module for every head. This is the next step toward the multi-head implementation commonly used in Transformers.

### 3.13 Building an Efficient Multi-Head Attention Layer

In the previous part, we created multiple attention heads by instantiating a separate `CausalSelfAttention` module for each head. That helped us understand the concept, but it isn't the most efficient implementation.

Now we'll build a more conventional Transformer-style implementation: **project the input into Query, Key, and Value representations once, then split those representations across multiple heads.**

#### 1. Understand the dimensionality

Suppose we have a batch of two sequences, each containing seven tokens. Each input token has three features.

<img src="./illustrations/3_13 Understand the dimensionality.png" width="700px">

Here are our dimensions:
- $ B = 2 $: batch size.
- $ T = 7 $: sequence length.
- $ D_{in} = 3 $: input features per token.
- $ D_{out} = 6 $: total features in each projected representation.
- $ H = 3 $: number of attention heads.
- $ d_{head} = D_{out}/H = 2 $: features assigned to each head.

We choose `d_out=6` because it divides evenly across our three heads. In general, the output dimension must be divisible by the number of heads.

#### 2. Project the input into Q, K and V

Each token is transformed into Query, Key and Value vectors. Unlike our earlier implementation, each projection produces six features, which we will divide among three heads.

In [81]:
import torch
import torch.nn as nn

d_in = 3
d_out = 6
num_heads = 3

# Each head receives an equal share of output features.
assert d_out % num_heads == 0
head_dim = d_out // num_heads

# Our existing batch has shape (2, 7, 3).
print("Input shape:", batch_inputs.shape)
print("Features per head:", head_dim)

Input shape: torch.Size([2, 7, 3])
Features per head: 2


The key idea is that we don't need three separate projection modules for every head. Each projection generates all six features, and we subsequently divide those features into three groups.

#### 3. Reshape the projected representations into heads

Let's examine the transformation using the Query projection.

In [82]:
# One projection transforms each token from 3 features to 6.
query_projection = nn.Linear(d_in, d_out, bias=False)

queries = query_projection(batch_inputs)
print("Before splitting:")
print("Queries:\n", queries)
print("Queries shape:", queries.shape)

# Split the 6 features into 3 heads, each with 2 features.
queries = queries.view(
    batch_inputs.shape[0], # Batch size
    batch_inputs.shape[1], # Sequence length
    num_heads,             # Number of heads
    head_dim,              # Features per head
)
print("\nAfter splitting:")
print("Queries:\n", queries)
print("Queries shape:", queries.shape)

# Move the head dimension before the sequence dimension.
queries = queries.transpose(1, 2)

print("\nAfter transposing:")
print("Queries:\n", queries)
print("Queries shape:", queries.shape)

Before splitting:
Queries:
 tensor([[[-0.3154, -0.1399,  0.3450,  0.4311, -0.1114, -0.3662],
         [-0.2154, -0.1495,  0.2049,  0.4725, -0.1701, -0.3194],
         [ 0.2018, -0.3007,  0.0880,  0.2885,  0.0699, -0.1212],
         [-0.2556, -0.1446,  0.3121,  0.3839, -0.0804, -0.3231],
         [ 0.0164, -0.1868,  0.0777,  0.3372, -0.0689, -0.1736],
         [ 0.0998, -0.3056,  0.2382,  0.2736,  0.1241, -0.1836],
         [ 0.0480, -0.2414,  0.1481,  0.3240,  0.0069, -0.1855]],

        [[-0.3154, -0.1399,  0.3450,  0.4311, -0.1114, -0.3662],
         [-0.2154, -0.1495,  0.2049,  0.4725, -0.1701, -0.3194],
         [ 0.2018, -0.3007,  0.0880,  0.2885,  0.0699, -0.1212],
         [-0.2556, -0.1446,  0.3121,  0.3839, -0.0804, -0.3231],
         [ 0.0164, -0.1868,  0.0777,  0.3372, -0.0689, -0.1736],
         [ 0.0998, -0.3056,  0.2382,  0.2736,  0.1241, -0.1836],
         [ 0.0480, -0.2414,  0.1481,  0.3240,  0.0069, -0.1855]]],
       grad_fn=<UnsafeViewBackward0>)
Queries shape: torch

Why do we transpose?
After splitting, the dimensions are:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ (B, T, H, d_{head} $

We want:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ (B, H, T, d_{head}) $

This puts the heads before the sequence dimension, making it convenient to calculate attention separately for every head using batched matrix multiplication.
We will perform the same projection, reshaping and transposition for Keys and Values.

#### 4. Implement the complete multi-head layer

In [83]:
class MultiHeadCausalSelfAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads):
        super().__init__()

        # Divide the total output features evenly among the heads.
        assert d_out % num_heads == 0

        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        # Project inputs into the combined Query, Key and Value representations.
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)

        # Combine the information produced by all heads.
        self.out_proj = nn.Linear(d_out, d_out, bias=False)

        self.dropout = nn.Dropout(dropout)

        # Mark future-token positions that causal attention must block.
        mask = torch.triu(
            torch.ones(context_length, context_length),
            diagonal=1
        )

        # Keep the mask with the model, but not as a trainable parameter.
        self.register_buffer("mask", mask)

    def forward(self, x):
        # Input shape: (batch_size, sequence_length, input_features)
        batch_size, sequence_length, _ = x.shape

        # Step 1: Project the input into Q, K, and V.
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        # Step 2: Split the features into separate heads.
        # Shape becomes (B, T, H, head_dim).
        queries = queries.view(
            batch_size, sequence_length, self.num_heads, self.head_dim
        )
        keys = keys.view(
            batch_size, sequence_length, self.num_heads, self.head_dim
        )
        values = values.view(
            batch_size, sequence_length, self.num_heads, self.head_dim
        )

        # Step 3: Arrange dimensions for attention calculations.
        # Shape becomes (B, H, T, head_dim).
        queries = queries.transpose(1, 2)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)

        # Step 4: Calculate attention scores independently for each head.
        # keys.transpose(2, 3) changes (B, H, T, head_dim)
        # into (B, H, head_dim, T)
        attention_scores = queries @ keys.transpose(2, 3)

        # Scale using the dimension of each head.
        attention_scores = attention_scores / (self.head_dim ** 0.5)

        # Step 5: Prevent each token from attending to future tokens.
        attention_scores = attention_scores.masked_fill(
            self.mask[:sequence_length, :sequence_length].bool(),
            -torch.inf
        )

        # Step 6: Convert scores into attention weights.
        attention_weights = torch.softmax(attention_scores, dim=-1)

        # Apply dropout to the attention weights during training.
        attention_weights = self.dropout(attention_weights)

        # Step 7: Retrieve weighted Valie information for each head.
        # Shape: (B, H, T, head_dim).
        context = attention_weights @ values

        # Step 8: Put the sequence dimension before the head dimension.
        # Shape: (B, T, H, head_dim)
        context = context.transpose(1, 2).contiguous()

        # Join the head features into one representation per token.
        # Shape: (B, T, d_out).
        context = context.view(batch_size, sequence_length, -1)

        # Step 9: Allow the model to mix information across heads.
        output = self.out_proj(context)

        return output

#### 5. Run the layer

In [84]:
torch.manual_seed(123)

multi_head_attention = MultiHeadCausalSelfAttention(
    d_in=3,
    d_out=6,
    context_length=7,
    dropout=0.2,
    num_heads=3
)

# Disable dropout for a predictable evaluation pass.
multi_head_attention.eval()

multi_head_output = multi_head_attention(batch_inputs)

print("Input:\n", batch_inputs)
print("Input shape:", batch_inputs.shape)
print("\nOutput:\n", multi_head_output)
print("Output shape:\n", multi_head_output.shape)

Input:
 tensor([[[0.4200, 0.1800, 0.7600],
         [0.8100, 0.3400, 0.5900],
         [0.6300, 0.9100, 0.2700],
         [0.3700, 0.2200, 0.6800],
         [0.7400, 0.5600, 0.3100],
         [0.2900, 0.7900, 0.4700],
         [0.5800, 0.6700, 0.3900]],

        [[0.4200, 0.1800, 0.7600],
         [0.8100, 0.3400, 0.5900],
         [0.6300, 0.9100, 0.2700],
         [0.3700, 0.2200, 0.6800],
         [0.7400, 0.5600, 0.3100],
         [0.2900, 0.7900, 0.4700],
         [0.5800, 0.6700, 0.3900]]])
Input shape: torch.Size([2, 7, 3])

Output:
 tensor([[[ 0.0588, -0.2150,  0.2750,  0.2179, -0.0213,  0.3021],
         [ 0.0366, -0.2400,  0.3052,  0.2491,  0.0029,  0.3273],
         [-0.0092, -0.2762,  0.2683,  0.2649,  0.0329,  0.2928],
         [ 0.0016, -0.2628,  0.2602,  0.2521,  0.0230,  0.2854],
         [-0.0082, -0.2635,  0.2565,  0.2533,  0.0310,  0.2801],
         [-0.0169, -0.2734,  0.2399,  0.2539,  0.0343,  0.2661],
         [-0.0233, -0.2785,  0.2354,  0.2567,  0.0386,  0.2618]

Each of the seven tokens in each sequence now has a six-dimensional contextual representation.

Notice that the output has six features rather than three. That is intentional: we configured d_out=6. Later, when we assemble the full Transformer, we will choose dimensions that fit the model's embedding size and residual connections.

#### What have we learned?

- **Separate heads:** Each head performs its own attention calculation.
- **Combined projections:** We project Q, K, and V into all head features, then split them.
- **Per-head scaling:** We divide scores by $ \sqrt{d_{head}} $, not by the square root of the total output dimension.
- **Output projection:** After concatenating the heads, the output projection learns how to combine their information.

Multiple heads give the model multiple learned attention pathways, but they are not guaranteed to specialize into distinct human-interpretable roles.

### 3.14 Understanding the Multi-Head Attention Implementation in *Build a Large Language Model From Scratch*

In section 3.13 , we built multi-head causal self-attention by projecting the input into Query, Key and Value representations, splitting the features into heads, calculating attention independently, and combining the results.

#### 1. Review the complete data flow

For our example, the input has shape (2, 7, 3): two sequences, seven tokens per sequence, and three input features per token.

<img src="./illustrations/3_14 Review the complete data flow.png" widht="700px">

The crucial point is that the heads operate in parallel on different learned feature subspaces. We don't process one head and then feed its output into the next head.

#### 2. Why do we need the output projection?

After attention, we concatenate the outputs from all heads. Each head contributes two features, so three heads produce six features per token.

Concatenation alone simply places the features side by side. It does not learn how to mix them.

The output projection is a learned linear transformation:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \text{Output = Concatenation heads} \space W_O $

Here, $ W_O $ has shape `(6, 6)` in our example.

In [85]:
# Use the output produced by our multi-head attention layer.
print("Multi-head output:\n", multi_head_output)
print("Multi-head output shape:", multi_head_output.shape)

# Inspect the learned output projection
print("\nOutput projection weight:\n", multi_head_attention.out_proj.weight)
print("Output projection weight shape:", multi_head_attention.out_proj.weight.shape)

Multi-head output:
 tensor([[[ 0.0588, -0.2150,  0.2750,  0.2179, -0.0213,  0.3021],
         [ 0.0366, -0.2400,  0.3052,  0.2491,  0.0029,  0.3273],
         [-0.0092, -0.2762,  0.2683,  0.2649,  0.0329,  0.2928],
         [ 0.0016, -0.2628,  0.2602,  0.2521,  0.0230,  0.2854],
         [-0.0082, -0.2635,  0.2565,  0.2533,  0.0310,  0.2801],
         [-0.0169, -0.2734,  0.2399,  0.2539,  0.0343,  0.2661],
         [-0.0233, -0.2785,  0.2354,  0.2567,  0.0386,  0.2618]],

        [[ 0.0588, -0.2150,  0.2750,  0.2179, -0.0213,  0.3021],
         [ 0.0366, -0.2400,  0.3052,  0.2491,  0.0029,  0.3273],
         [-0.0092, -0.2762,  0.2683,  0.2649,  0.0329,  0.2928],
         [ 0.0016, -0.2628,  0.2602,  0.2521,  0.0230,  0.2854],
         [-0.0082, -0.2635,  0.2565,  0.2533,  0.0310,  0.2801],
         [-0.0169, -0.2734,  0.2399,  0.2539,  0.0343,  0.2661],
         [-0.0233, -0.2785,  0.2354,  0.2567,  0.0386,  0.2618]]],
       grad_fn=<UnsafeViewBackward0>)
Multi-head output shape: tor

The projection can learn combinations of features from different heads. It doesn't guarantee that every head contributes equally, or that each head learns a distinct linguistic function.

#### 3. Why is the scaling factor based on `head_dim`?

Recall the attention equation:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \text{Attention}(Q,K,V) = \text{softmax} \left(\frac{QK^T}{\sqrt{d_k}}\right)V $

In multi-head attention, each head uses its own Query and Key vectors. Therefore, $ d_k $ is the number of features **per head**, not the combined feature count across all heads.

For our example:
- Total projected features: `d_out = 6`
- Number of heads: `num_heads = 3`
- Features per head: `head_dim = 2`
- Correct scaling factor: $ \sqrt{2} $

If we incorrectly divided by $ \sqrt{6} $, we would be scaling the scores using the combined dimension rather than the dimension used to compute each head's dot products.

In [86]:
import math

print("Features per head:\t\t      ", multi_head_attention.head_dim)
print("Correct scaling factor:\t\t      ", math.sqrt(multi_head_attention.head_dim))
print("Incorrect whole-output scaling factor:", math.sqrt(6))

Features per head:		       2
Correct scaling factor:		       1.4142135623730951
Incorrect whole-output scaling factor: 2.449489742783178


#### 4. Check the causal mask across all heads

Each head must obey the same causal rule: a token cannot attend to future tokens. The mask has shape `(7, 7)` and broadcasts across the batch and head dimensions.

For a seven-token sequence, the attention score tensor has shape `(2, 3, 7, 7)`. The first two dimensions identify the batch item and head; the final two dimensions represent query positions and key positions.

In [87]:
# Inspect the causal mask registered in the module.
print(multi_head_attention.mask)

tensor([[0., 1., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0.]])


The upper-triangular positions are blocked with negative infinity before softmax. Their resulting attention weights are zero, preventing information from future tokens from influencing earlier token representations.

### 3.15 Final Verification — Does Our Attention Mechanism Work?

Before we close this module, let's verify the attention mechanism we've built. So far, we've implemented causal masking, dropout, batching and multi-head attention. Now we'll test the mathematical properties that make the mechanism work correctly.

We'll use the `MultiHeadCausalSelfAttention` class from Section 3.13.

#### 1. Verify the output shape

Our input has shape $ ((B,T,D_{in})=(2,7,3) $. We configured the attention layer with three heads and six output features.

Therefore, the expected output shape is $ ((2,7,6) $.

In [88]:
# Put the model in evaluation mode to disable dropout.
multi_head_attention.eval()

with torch.no_grad():
    output = multi_head_attention(batch_inputs)

print("Input:\n", batch_inputs)
print("Input shape:", batch_inputs.shape)
print("\nOutput:\n", output)
print("Output shape:", output.shape)

assert output.shape == (2, 7, 6)
print("\nOutput shape verified.")

Input:
 tensor([[[0.4200, 0.1800, 0.7600],
         [0.8100, 0.3400, 0.5900],
         [0.6300, 0.9100, 0.2700],
         [0.3700, 0.2200, 0.6800],
         [0.7400, 0.5600, 0.3100],
         [0.2900, 0.7900, 0.4700],
         [0.5800, 0.6700, 0.3900]],

        [[0.4200, 0.1800, 0.7600],
         [0.8100, 0.3400, 0.5900],
         [0.6300, 0.9100, 0.2700],
         [0.3700, 0.2200, 0.6800],
         [0.7400, 0.5600, 0.3100],
         [0.2900, 0.7900, 0.4700],
         [0.5800, 0.6700, 0.3900]]])
Input shape: torch.Size([2, 7, 3])

Output:
 tensor([[[ 0.0588, -0.2150,  0.2750,  0.2179, -0.0213,  0.3021],
         [ 0.0366, -0.2400,  0.3052,  0.2491,  0.0029,  0.3273],
         [-0.0092, -0.2762,  0.2683,  0.2649,  0.0329,  0.2928],
         [ 0.0016, -0.2628,  0.2602,  0.2521,  0.0230,  0.2854],
         [-0.0082, -0.2635,  0.2565,  0.2533,  0.0310,  0.2801],
         [-0.0169, -0.2734,  0.2399,  0.2539,  0.0343,  0.2661],
         [-0.0233, -0.2785,  0.2354,  0.2567,  0.0386,  0.2618]

**Why this matters:** the layer must preserve the batch and sequence dimensions while producing the configured number of output features.

#### 2. Verify that causal masking works

A token must not attend to any token that comes after it. We can test this by calculating the attention weights directly and checking that every future-token position has a weight of zero.

Remember that the causal mask is applied before softmax, so the blocked scores become negative infinity. Softmax converts these scores into zero weights.

In [89]:
with torch.no_grad():
    # Recreate the attention calculations for inspection.
    x = batch_inputs

    B, T, _ = x.shape

    queries = multi_head_attention.W_query(x)
    keys = multi_head_attention.W_key(x)

    # Split into heads
    queries = queries.view(B, T, 3, 2).transpose(1, 2)
    keys = keys.view(B, T, 3, 2).transpose(1, 2)

    # Calculate and scale the attention scores.
    scores = queries @ keys.transpose(2, 3)
    scores = scores / (multi_head_attention.head_dim ** 0.5)

    # Block future positions.
    scores = scores.masked_fill(
        multi_head_attention.mask[:T, :T].bool(),
        -torch.inf
    )

    # Convert scores into weights.
    weights = torch.softmax(scores, dim=-1)

    # Select all positions strictly above the diagonal.
    future_weights = weights[
        :, :, torch.triu(torch.ones(T, T), diagonal=1).bool()
    ]

    print("Largest future-token weight:", future_weights.max().item())

    assert torch.all(future_weights == 0)
    print("Causal masking verified.")

Largest future-token weight: 0.0
Causal masking verified.


**What we expect:** the largest future-token weight is exactly 0.0.

This verifies the mask at the attention-weight level, rather than merely checking that a mask exists.

#### 3. Verify that attention weights sum to one

Before dropout, softmax converts each row of attention scores into a probability distribution. Each query position's weights across all key positions must sum to one.

Causal masking doesn't change this property: the blocked positions have zero weight, and the permitted positions share the total probability mass.

In [90]:
# Each row corresponds to one query token.
# The last dimension contains its weights over key positions.
row_sums = weights.sum(dim=-1)
print("Attention weight sums:\n", row_sums)

assert torch.allclose(
    row_sums,
    torch.ones_like(row_sums),
    atol=1e-6
)

print("Attention weight normalization verified.")

Attention weight sums:
 tensor([[[1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000]],

        [[1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000]]])
Attention weight normalization verified.


The output should contain values very close to `1.0` for every batch item, head and query position.

Note that this check is performed **before dropout**. During training, dropout can zero some weights and scale others, so the resulting rows do not necessarily sum to one.

#### 4. Verify that gradients can flow through the layer

Attention is useful in a trainable language model because its projection weights can be updated through backpropagation.

We'll run a small training-style operation and check that gradients are produced for the learned Query, Key and Value projections.

In [91]:
# Return to training mode.
multi_head_attention.train()

# Clear gradients left over from any previous training step.
multi_head_attention.zero_grad()

# Run a forward pass.
output = multi_head_attention(batch_inputs)

# Create a simple scalar objective for this demonstration.
loss = output.square().mean()

# Backpropagation through the attention mechanism.
loss.backward()

# Inspect the projection gradients.
for name, parameter in multi_head_attention.named_parameters():
    if parameter.grad is not None:
        print(
            name,
            "gradient exists:",
            parameter.grad is not None,
            "   gradient magnitude:",
            parameter.grad.abs().mean().item()
        )

W_query.weight gradient exists: True    gradient magnitude: 0.0002832022728398442
W_key.weight gradient exists: True    gradient magnitude: 0.0002883979177568108
W_value.weight gradient exists: True    gradient magnitude: 0.026816483587026596
out_proj.weight gradient exists: True    gradient magnitude: 0.0187202300876379


**What this verifies:** the loss can propagate gradients through the attention computations into the learned projection and output weights.

This is a gradient-flow test, not a complete training experiment. We haven't trained a language model or established that the attention layer has learned useful language patterns.

### Module 3 complete

You've now built and tested the main components of the attention mechanism:
- Token representations and dot-product attention scores.
- Softmax attention weights and weighted context vectors.
- Learned Query, Key and Value projections.
- Scaled dot-product attention.
- Causal masking and dropout.
- Batched self-attention.
- Multi-head attention with combined projections and an output projection.
- Shape, masking, normalization and gradient-flow verification.
  
The central equation behind the mechanism is:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $ \text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V $

For causal language modelling, we additionally mask future positions before applying softmax.

**Next module:** we'll move beyond attention itself and begin assembling the larger Transformer architecture, starting with how attention works alongside residual connections, normalization and feed-forward layers.